In [1]:
# ── 1. Install dependencies ────────────────────────────────────────────────
!pip install -q kaggle xgboost lightgbm scikit-learn langgraph python-frontmatter shap
!pip install -q openai
!pip install langchain-openai langchain langgraph

In [2]:
# ── 2. Imports & PipelineState ────────────────────────────────────────────
import os, json, random, glob, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Any, Dict, List, Optional
from typing_extensions import TypedDict

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
    os.environ["OPENAI_API_KEY"]  = userdata.get("OPENAI_API_KEY")
    print("API keys loaded from Colab secrets.")
except Exception:
    import getpass
    if "KAGGLE_USERNAME" not in os.environ:
        os.environ["KAGGLE_USERNAME"] = getpass.getpass("Kaggle username: ")
    if "KAGGLE_KEY" not in os.environ:
        os.environ["KAGGLE_KEY"]      = getpass.getpass("Kaggle key: ")
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"]  = getpass.getpass("OpenAI API key: ")
    print("API keys set.")


class PipelineState(TypedDict):
    inference_station_id: str
    inference_hour:       int
    inference_dow:        int
    inference_month:      int
    # Training configuration (pass at invocation)
    use_asymmetric_loss:  bool          # True -> QuantileReg + asym XGB/LGBM; False -> Ridge + sym MSE
    alpha_grid:           List[float]   # e.g. [1.5, 2.0, 3.0]; ignored when use_asymmetric_loss=False
    X_train: Any;  X_test:  Any
    y_train: Any;  y_test:  Any
    feature_names: List[str]
    capacity_map:  Dict[str, int]
    lat_map:       Dict[str, float]
    lon_map:       Dict[str, float]
    trained_models: Dict[str, Any]
    evaluation_results: Dict[str, Dict[str, float]]
    eval_plot_path:     str
    selected_model:        str
    selected_model_object: Any
    selection_reason:      str
    prediction:            float
    prediction_confidence: str
    confidence_note:       str
    current_inventory:     float
    dispatch_decision:  str
    dispatch_reasoning: str
    business_output:    str


API keys loaded from Colab secrets.


In [3]:
# ── 3. Define & write SKILL.md files ──────────────────────
import os
os.makedirs("skills", exist_ok=True)

skills = {

"skills/preprocess-data.md": """\
---
name: preprocess-data
description: >
  Use when preparing raw Citi Bike CSV data for model training.
  Triggers on: data cleaning, resampling, feature engineering,
  train/test split, missing value imputation.
mode: organisational
---

# Preprocess Data Skill

## Role
Organisational skill. Pure Python,no LLM call.

## When to Use
Invoke at the start of every pipeline run before any model training occurs.
Triggers on: raw Citi Bike CSV data available, data cleaning required,
feature engineering needed, train/test split needed.

## How to Execute
1. Download 5 CSV files from Kaggle (random seed=42)
2. Filter to 2021 only; keep active stations (is_installed=1, is_renting=1)
3. Resample to 30-min intervals per station (mean aggregation); carry lat/lon
4. Drop stations with >30% missing rate after resampling
5. Forward-fill + backward-fill remaining NaNs per station
6. Feature engineering: hour, day_of_week, month, cyclic encodings,
   is_peak, is_weekend, is_covid, lag_1/2/3/6/48/336,
   rolling_mean_6, rolling_std_6, availability_ratio, bikes_current, lat, lon
7. 80/20 chronological split -> write to state

## Inputs from agent state
This is the pipeline entry node. No upstream state inputs are required.
Reads use_asymmetric_loss and alpha_grid from the initial invocation state.

## Outputs to agent state
- X_train, X_test  (pd.DataFrame)
- y_train, y_test  (pd.Series) — target: num_bikes_available at t+1
- feature_names    (List[str])
- capacity_map     (Dict[str, int])
- lat_map, lon_map (Dict[str, float])

## Output format
pandas objects stored directly in state

## Notes
- lag_336 requires 7 days of burn-in per station; first week rows dropped via dropna
- is_covid = 1 for Jan–Jun 2021 (restricted ridership), 0 for Jul–Dec 2021
- bikes_current = lag_0 (current slot observation), renamed to distinguish from target
""",

"skills/train-models.md": """\
---
name: train-models
description: >
  Use when training candidate models on preprocessed Citi Bike data.
  Triggers on: model training, fit regressor, quantile regressor, XGBoost, LightGBM, Ridge.
mode: organisational
---

# Train Models Skill

## Role
Organisational skill. Pure Python, no LLM call.

## When to Use
Invoke immediately after preprocess-data has written X_train and y_train to state.
Triggers on: training data available in state, candidate models need to be fitted,
asymmetric or symmetric loss configuration required.

## How to Execute
1. Read use_asymmetric_loss and alpha_grid from state.
2. If use_asymmetric_loss is True:
   a. For each alpha in alpha_grid, fit XGBRegressor and LGBMRegressor with a
      custom asymmetric gradient objective (overestimation penalised alpha x more).
   b. Fit one QuantileRegressor with q = 1 / (1 + alpha) for the primary alpha.
   c. Key name format: xgboost_a{alpha}, lightgbm_a{alpha}, ridge (quantile).
3. If use_asymmetric_loss is False:
   a. Fit one Ridge regressor, one XGBRegressor, one LGBMRegressor with default MSE.
   b. Key names: ridge, xgboost_a1.0, lightgbm_a1.0.
4. Attempt GPU training (cuda/gpu); fall back to CPU silently on failure.
5. Write all fitted model objects to state["trained_models"].

### Candidate Models
| Key               | Class               | Loss mode          |
|-------------------|---------------------|--------------------|
| ridge             | Ridge / QuantileReg | MSE or quantile    |
| xgboost_a{alpha}  | XGBRegressor        | Asymmetric or MSE  |
| lightgbm_a{alpha} | LGBMRegressor       | Asymmetric or MSE  |

### Hyperparameters (shared across XGB and LGBM)
- n_estimators=500, learning_rate=0.03, max_depth=6
- subsample=0.8, colsample_bytree=0.8, random_state=42

## Inputs from agent state
- X_train (DataFrame)   — from preprocess-data
- y_train (Series)      — from preprocess-data
- use_asymmetric_loss (bool)   — from initial invocation state
- alpha_grid (List[float])     — from initial invocation state

## Outputs to agent state
- trained_models (Dict[str, model]) — all fitted model objects keyed by name

## Output format
Python dict stored directly in state["trained_models"].
Example keys: {"ridge": Ridge, "xgboost_a1.0": XGB, "xgboost_a3.0": XGB,
               "lightgbm_a1.0": LGBM, "lightgbm_a3.0": LGBM}

## Notes
- The _a{alpha} suffix convention is required by select-model to identify the
  symmetric baseline (alpha=1.0) and the highest-penalty variant.
- GPU training is attempted silently and falls back to CPU; no manual flag needed.
- No evaluation happens here; all metric computation is deferred to evaluate-models.

""",

"skills/evaluate-models.md": """\
---
name: evaluate-models
description: >
  Use when evaluating trained models on the test split and peak-hour subset.
  Triggers on: RMSE/MAE/R2 reporting, overestimation analysis, residual plotting.
mode: organisational
---

# Evaluate Models Skill

## Role
Organisational skill. Pure Python, no LLM call.

## When to Use
Invoke immediately after train-models has written all fitted models to state.
Triggers on: trained_models available in state, metric reporting needed,
evaluation visualisations required, model comparison about to begin.

## How to Execute
1. Read trained models from state["trained_models"].
2. Predict on X_test and compute full-set metrics per model:
   - rmse, mae, r2, overestimation_rate
3. Compute peak-hour subset metrics (is_peak == 1):
   - peak_rmse, peak_mae, peak_overestimation_rate
4. Generate diagnostic plots for each model:
   - Predicted vs Actual scatter plot
   - Residual plot (Pred - Actual)
5. Save figure path into state.

## Inputs from agent state
- trained_models (Dict[str, model])
- X_test (pd.DataFrame)
- y_test (pd.Series)

## Outputs to agent state
- evaluation_results (Dict[str, Dict[str, float]])
- eval_plot_path (str)

## Output format
metrics dictionary + local image path

## Notes
- If no peak rows exist, peak metrics should be NaN.
- Overestimation rate = mean(prediction > actual).
""",

"skills/select-model.md": """\
---
name: select-model
description: >
  Use when selecting the final model under asymmetric business risk.
  Triggers on: baseline filtering, overestimation minimization, fallback decisions, business explanation generation.
mode: llm
---

# Select Model Skill

## Role
Hybrid skill: deterministic model selection logic + LLM business summary.

## When to Use
Invoke immediately after evaluate-models has written evaluation_results to state.
Triggers on: model metrics available, final model selection needed,
business justification for model choice required.

## How to Execute
1. Measure inference speed (ms per 1000 rows, averaged across 5 runs).
2. Build symmetric baseline from alpha=1.0 models:
   - baseline_rmse = minimum RMSE among models tagged with _a1.0
3. Apply RMSE filter:
   - keep models with RMSE <= baseline_rmse * 1.15
4. Primary selection rule:
   - if filtered set contains any model at the highest alpha in alpha_grid, choose
     the model with the lowest overestimation_rate among all filtered models
5. Fallback rule:
   - if no highest-alpha model survives filtering, emit warning and choose the
     model with lowest peak_rmse
6. Secondary fallback:
   - if no alpha=1.0 model exists, emit warning and choose lowest peak_rmse
7. Use OpenAI to generate 2-4 sentence non-technical business explanation.

## Inputs from agent state
- trained_models (Dict[str, model])
- evaluation_results (Dict[str, Dict[str, float]])
- X_test (pd.DataFrame)

## Outputs to agent state
- selected_model (str)
- selected_model_object (Any)
- selection_reason (str)

## Output format
final selected model id + short business rationale

## Notes
- Model name suffix format uses _a{alpha}, e.g. xgboost_a3.0.
- Fallback warning should clearly indicate which max alpha value had no survivors.
""",
"skills/run-inference.md": """\
---
name: run-inference
description: >
  Use when the selected model object needs to produce a numeric demand forecast
  for a specific station. Triggers on: model selected, inference input available,
  predict bikes available at t+1.
mode: organisational
---

# Run Inference Skill

## Role
Organisational skill. Pure Python — no LLM call.

## When to Use
Invoke immediately after select-model has written selected_model_object to state.
This node runs the trained model on a real input row and produces a numeric
prediction that downstream agent skills can act on.

## How to Execute
1. Read selected_model_object, X_test, feature_names, lat_map, lon_map from state.
2. Look up inference_station_id in lat_map/lon_map to get target (lat, lon).
3. Spatial search: find the closest X_test row to target coordinates via Euclidean distance.
4. Temporal alignment: filter matched rows by inference_hour, inference_dow, inference_month;
   fall back to latest spatial match if no exact temporal match exists.
5. Call model.predict(sample_features) on the matched row and clamp result to >= 0.
6. Read rolling_std_6 from the matched row to compute prediction confidence:
   - rolling_std_6 < 2  → High confidence
   - rolling_std_6 < 4  → Medium confidence
   - rolling_std_6 >= 4 → Low confidence
5. Write prediction, prediction_confidence, and confidence_note to state.

## Inputs from Agent State
| Field                  | Type            | Source              |
|------------------------|-----------------|---------------------|
| selected_model_object  | Any             | select-model        |
| X_test                 | DataFrame       | preprocess-data     |
| feature_names          | List[str]       | preprocess-data     |
| lat_map                | Dict[str,float] | preprocess-data     |
| lon_map                | Dict[str,float] | preprocess-data     |
| capacity_map           | Dict[str,int]   | preprocess-data     |
| inference_station_id   | str             | Gradio / user input |
| inference_hour         | int             | Gradio / user input |
| inference_dow          | int             | Gradio / user input |
| inference_month        | int             | Gradio / user input |

## Outputs to Agent State
| Field                 | Type  | Description                                      |
|-----------------------|-------|--------------------------------------------------|
| prediction            | float | Predicted bikes available at t+1 (clamped >= 0)  |
| prediction_confidence | str   | One of: "High", "Medium", "Low"                  |
| confidence_note       | str   | Plain-language explanation of confidence level   |

## Output Format
Numeric float written directly to state["prediction"].
Example: prediction=12.47, prediction_confidence="High",
confidence_note="Consistent demand pattern over the past 6 hours."

## Notes
- Uses spatial-temporal lookup in X_test as a proxy for real-time inference.
  In production this would be replaced by a live station telemetry feed.
- Fallback prediction of 5.0 bikes is used if model or test data is missing.
- rolling_std_6 measures demand variability over the last 6 time steps (90 min).
  High variability reduces confidence and triggers shorter recheck intervals downstream.
""",

"skills/dispatch-advisor.md": """\
---
name: dispatch-advisor
description: >
  Use when a predicted bike availability figure has been produced by run-inference
  and an operations team needs to know what action to take at the station.
  Triggers on: demand forecast received, station health evaluation, routing decision,
  inventory triage, bike rebalancing, surge event handling.
mode: llm-agent
---

# Dispatch Advisor Skill

## Role
LLM agent skill. Uses GPT-4-turbo with two pre-fetched tool results —
adj_stations_tool and event_calendar_tool — to evaluate station health
and produce a routing decision: REBALANCE, ESCALATE, or MONITOR.

## When to Use
Invoke immediately after run-inference has written a numeric demand
prediction to agent state. This skill bridges the gap between a raw model
forecast and an actionable field instruction for a non-technical dispatcher
or operations manager.

## How to Execute
1. Read prediction, inference_station_id, capacity_map, and
   use_asymmetric_loss from state.
2. Derive current_inventory.
3. Compute availability_ratio = current_inventory / capacity.
4. Call adj_stations_tool(station_id) to discover nearby surplus stations.
5. Call event_calendar_tool(location, date) to check for surge-risk events.
6. Invoke the LLM with station context, pre-fetched tool results, and decision rules:
   - availability_ratio < 0.20 → ESCALATE (critically low, manual override needed)
   - prediction > current_inventory × 1.5 → REBALANCE (demand surge forecast)
   - otherwise → MONITOR (steady state, no immediate action)
7. Parse the agent's structured response to extract dispatch_decision and
   dispatch_reasoning.
8. Write both fields back to agent state for downstream terminal nodes.

## Decision Rules
| Condition                                              | Decision  |
|--------------------------------------------------------|-----------|
| availability_ratio < 0.20 (critically low stock)      | ESCALATE  |
| prediction > current_inventory × 1.5 (forecast surge) | REBALANCE |
| Neither condition met                                  | MONITOR   |

## Inputs from Agent State
| Field                 | Type           | Source              |
|-----------------------|----------------|---------------------|
| prediction            | float          | run-inference       |
| prediction_confidence | str            | run-inference       |
| confidence_note       | str            | run-inference       |
| inference_station_id  | str            | Gradio / user input |
| capacity_map          | Dict[str, int] | preprocess-data     |
| use_asymmetric_loss   | bool           | Gradio / user input |

## Outputs to Agent State
| Field              | Type | Description                                              |
|--------------------|------|----------------------------------------------------------|
| dispatch_decision  | str  | One of: "REBALANCE", "ESCALATE", "MONITOR"               |
| dispatch_reasoning | str  | Plain-language explanation (2-4 sentences) for dispatcher|

## Output Format
{
  "dispatch_decision": "REBALANCE",
  "dispatch_reasoning": "The model predicts 23 bikes needed in the next 30 minutes
    but only 15 are currently available. Station JC105 has a surplus and is 300m away.
    A bike transfer is recommended before the morning rush peak."
}

## Routing Logic (dispatch_router)
The conditional edge dispatch_router reads state["dispatch_decision"] and
routes to one of three terminal nodes:
- "REBALANCE" → rebalance node
- "ESCALATE"  → escalate node
- default     → monitor node

## Notes
- The event_calendar_tool uses deterministic seed generation (f"{date}_{location}")
  so the same query always returns the same event status, ensuring reproducibility.
- If capacity_map does not contain inference_station_id, capacity defaults to 30.
- If the LLM fails to produce a parseable decision, default to "MONITOR"
  to avoid false escalations.
- prediction_confidence informs the reasoning text but does not override the
  rule-based routing logic.
""",

"skills/rebalance.md": """\
---
name: rebalance
description: >
  Use when dispatch-advisor has determined that a station faces an imminent
  demand surge and bikes must be transferred from a nearby surplus station.
  Triggers on: dispatch_decision == "REBALANCE", forecast demand exceeds
  current inventory by more than 50%.
mode: llm-agent
---

# Rebalance Skill

## Role
LLM agent skill. Terminal node. Generates a concrete rebalancing work order
in plain business language for a field dispatcher or operations team member
to act on immediately. Ends the pipeline by writing business_output to state.

## When to Use
Invoked automatically by dispatch_router when dispatch_decision equals
"REBALANCE". Do not invoke directly — this node is wired as a terminal
conditional branch in the LangGraph pipeline.

## How to Execute
1. Read dispatch_reasoning, prediction, inference_station_id,
   prediction_confidence, selected_model, and selection_reason from state.
2. Construct a dispatcher-facing work order that includes:
   - The station needing bikes.
   - The nearest surplus source station (from dispatch_reasoning context).
   - The number of bikes to transfer (derived from prediction − current_inventory).
   - Time urgency based on prediction_confidence:
     High → act within 10 min / Medium → act within 20 min / Low → verify first.
3. Format the output as a structured plain-English field instruction.
4. Write the final result to state["business_output"].

## Inputs from Agent State
| Field                 | Type  | Source              |
|-----------------------|-------|---------------------|
| dispatch_reasoning    | str   | dispatch-advisor    |
| dispatch_decision     | str   | dispatch-advisor    |
| prediction            | float | run-inference       |
| prediction_confidence | str   | run-inference       |
| confidence_note       | str   | run-inference       |
| inference_station_id  | str   | Gradio / user input |
| selected_model        | str   | select-model        |
| selection_reason      | str   | select-model        |

## Outputs to Agent State
| Field           | Type | Description                                              |
|-----------------|------|----------------------------------------------------------|
| business_output | str  | Complete plain-language rebalancing work order for Gradio|

## Output Format
🚲 REBALANCING WORK ORDER
Station: {station_id}
Action Required: Transfer {n} bikes from surplus station {source_id}
Urgency: {High / Medium / Low} — act within {10 / 20 / review} minutes
Reason: {dispatch_reasoning}
Prediction Confidence: {confidence} — {confidence_note}
Powered by: {selected_model}

## Notes
- A business user reading this output should need no knowledge of the ML model,
  probability scores, or LangGraph internals.
- The number of bikes to transfer should be clamped to physical vehicle capacity
  and the surplus available at the source station.
- This node writes to business_output and terminates at END.
- The Gradio interface displays business_output as the primary result panel.
""",

"skills/escalate.md": """\
---
name: escalate
description: >
  Use when dispatch-advisor has determined that a station has critically low
  bike availability (below 20% of capacity) and the situation exceeds automated
  rebalancing thresholds. Triggers on: dispatch_decision == "ESCALATE",
  availability_ratio < 0.20, potential service failure risk.
mode: llm-agent
---

# Escalate Skill

## Role
LLM agent skill. Terminal node. Generates a critical alert report for a senior
operations manager or on-call supervisor. Escalation indicates that automated
rebalancing is insufficient and human judgement or emergency response is needed.
Ends the pipeline by writing business_output to state.

## When to Use
Invoked automatically by dispatch_router when dispatch_decision equals
"ESCALATE". Occurs when availability_ratio < 0.20 — a service failure risk
that cannot be resolved by a simple bike transfer alone.

## How to Execute
1. Read dispatch_reasoning, prediction, inference_station_id,
   prediction_confidence, confidence_note, selected_model from state.
2. Determine severity from prediction_confidence:
   - High confidence + critically low inventory → Severity: CRITICAL
   - Medium confidence + critically low inventory → Severity: HIGH
   - Low confidence + critically low inventory  → Severity: ELEVATED (verify first)
3. Generate a structured escalation alert naming the station, current stock,
   forecast demand, shortfall, and recommended supervisor action.
4. Write the complete alert to state["business_output"].

## Inputs from Agent State
| Field                 | Type  | Source              |
|-----------------------|-------|---------------------|
| dispatch_reasoning    | str   | dispatch-advisor    |
| dispatch_decision     | str   | dispatch-advisor    |
| prediction            | float | run-inference       |
| prediction_confidence | str   | run-inference       |
| confidence_note       | str   | run-inference       |
| inference_station_id  | str   | Gradio / user input |
| selected_model        | str   | select-model        |
| selection_reason      | str   | select-model        |

## Outputs to Agent State
| Field           | Type | Description                                              |
|-----------------|------|----------------------------------------------------------|
| business_output | str  | Structured escalation alert for Gradio display           |

## Output Format
🚨 ESCALATION ALERT — CRITICAL STATION STATUS
Station: {station_id}
Severity: {CRITICAL / HIGH / ELEVATED}
Current Stock: CRITICALLY LOW (below 20% capacity)
Forecast Demand: {prediction:.0f} bikes needed in next 30 minutes
Shortfall Estimate: {shortfall} bikes
Recommended Action: Contact on-call supervisor immediately. Emergency restock required.
Context: {dispatch_reasoning}
Prediction Confidence: {confidence} — {confidence_note}
⚠️ This alert requires human review. Do not rely solely on automated routing.
Powered by: {selected_model}

## Notes
- Escalation is the highest-priority terminal state. The Gradio interface should
  render this output with a visible red/warning indicator.
- This node surfaces the problem for human action and does not attempt to solve it.
  Critically low availability may have causes (vandalism, blocked docks, system outage)
  that a rebalancing order cannot fix.
- confidence_note is displayed verbatim so the supervisor can judge whether to trust
  the alert or verify on the ground first.
- This node writes to business_output and terminates at END.
""",

"skills/monitor.md": """\
---
name: monitor
description: >
  Use when dispatch-advisor has determined that a station is operating within
  normal parameters and no immediate intervention is required.
  Triggers on: dispatch_decision == "MONITOR", demand forecast within acceptable
  inventory range, steady-state station health.
mode: organisational
---

# Monitor Skill

## Role
Organisational skill. Terminal node. Produces a plain-language station health
summary confirming that no action is needed at this time, and provides the
dispatcher with a situational awareness snapshot for the next 30-minute window.
Ends the pipeline by writing business_output to state.

## When to Use
Invoked automatically by dispatch_router when dispatch_decision equals
"MONITOR" — the default outcome when neither a rebalancing threshold nor a
critical inventory threshold has been breached. This is the most common terminal
state under normal operating conditions.

## How to Execute
1. Read dispatch_reasoning, prediction, inference_station_id,
   prediction_confidence, confidence_note, selected_model from state.
2. Compute projected_remaining = current_inventory - prediction (clamped to 0).
3. Determine next-check urgency from prediction_confidence:
   - High   → Next check in 30 minutes
   - Medium → Next check in 15 minutes
   - Low    → Next check in 10 minutes (uncertainty is high)
4. Generate a concise station health digest in plain English.
5. Write the digest to state["business_output"].

## Inputs from Agent State
| Field                 | Type  | Source              |
|-----------------------|-------|---------------------|
| dispatch_reasoning    | str   | dispatch-advisor    |
| dispatch_decision     | str   | dispatch-advisor    |
| prediction            | float | run-inference       |
| prediction_confidence | str   | run-inference       |
| confidence_note       | str   | run-inference       |
| inference_station_id  | str   | Gradio / user input |
| selected_model        | str   | select-model        |
| selection_reason      | str   | select-model        |

## Outputs to Agent State
| Field           | Type | Description                                              |
|-----------------|------|----------------------------------------------------------|
| business_output | str  | Plain-language station health digest for Gradio display  |

## Output Format
✅ STATION STATUS: NO ACTION REQUIRED
Station: {station_id}
Current Stock: Normal
Forecast Demand: {prediction:.0f} bikes in next 30 minutes
Projected Remaining After Demand: {projected_remaining:.0f} bikes
Confidence: {confidence} — {confidence_note}
Next Recommended Check: {30 / 15 / 10} minutes
Context: {dispatch_reasoning}
Powered by: {selected_model}

## Notes
- MONITOR is the most frequent terminal outcome. The Gradio interface renders
  this with a neutral green indicator (vs amber for REBALANCE, red for ESCALATE).
- projected_remaining gives the dispatcher advance notice of whether the station
  may need attention in the next check cycle.
- When prediction_confidence is "Low", the next-check interval is shortened to
  10 minutes even in MONITOR state, because high uncertainty may mean the
  steady-state assessment is wrong.
- This node is pure Python formatting — no LLM or external tool call is made.
  The routing decision has already been made by dispatch-advisor.
- This node writes to business_output and terminates at END.
""",
}  # end skills dict

for path, content in skills.items():
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
print("Skills written:", list(skills.keys()))


Skills written: ['skills/preprocess-data.md', 'skills/train-models.md', 'skills/evaluate-models.md', 'skills/select-model.md', 'skills/run-inference.md', 'skills/dispatch-advisor.md', 'skills/rebalance.md', 'skills/escalate.md', 'skills/monitor.md']


In [4]:
# ── 4. SkillStore ─────────────────────────────────────────
import frontmatter

class SkillStore:
    def __init__(self, skills_dir='skills'):
        self.skills_dir = Path(skills_dir)
        self._index = {}
        self._build_index()

    def _build_index(self):
        for md_file in self.skills_dir.glob('*.md'):
            post = frontmatter.load(str(md_file))
            name = post.metadata.get('name', md_file.stem)
            self._index[name] = {
                'path':        md_file,
                'description': post.metadata.get('description', ''),
                'mode':        post.metadata.get('mode', 'organisational'),
            }
        print(f'[SkillStore] Indexed {len(self._index)} skills: {list(self._index.keys())}')

    def get_content(self, skill_name: str) -> str:
        if skill_name not in self._index:
            print(f'[SkillStore] WARNING: skill "{skill_name}" not found, using fallback.')
            return (
                f'You are a Citi Bike operations assistant responsible for the '
                f'"{skill_name}" step. Apply best-practice operations logic '
                f'and return a clear, actionable response.'
            )
        try:
            post = frontmatter.load(str(self._index[skill_name]['path']))
            return post.content
        except Exception as e:
            print(f'[SkillStore] WARNING: failed to load skill "{skill_name}": {e}')
            return (
                f'You are a Citi Bike operations assistant responsible for the '
                f'"{skill_name}" step. Apply best-practice operations logic '
                f'and return a clear, actionable response.'
            )

    def get_mode(self, skill_name: str) -> str:
        return self._index.get(skill_name, {}).get('mode', 'organisational')

skill_store = SkillStore()

[SkillStore] Indexed 9 skills: ['monitor', 'evaluate-models', 'select-model', 'run-inference', 'dispatch-advisor', 'rebalance', 'preprocess-data', 'train-models', 'escalate']


In [5]:
# ── 5. preprocess-data node ────────────────────────────────────────────────
def preprocess_data(state: PipelineState) -> PipelineState:
    skill_name = 'preprocess-data'
    print(f'\n[preprocess-data] Skill mode: {skill_store.get_mode(skill_name)}')

    # Download CSVs
    os.makedirs('data', exist_ok=True)
    all_csvs = [f'citi_bike_data_{i:05d}.csv' for i in range(49)]
    random.seed(42)
    sampled = random.sample(all_csvs, 5)
    print('[preprocess-data] Sampled:', sampled)
    for fname in sampled:
        os.system(f'kaggle datasets download -d rosenthal/citi-bike-stations -f {fname} -p data/ --unzip -q')
    for zf_path in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zf_path) as zf:
            zf.extractall('data/')
        os.remove(zf_path)
    local_files = sorted(glob.glob('data/*.csv'))

    # Load CSVs
    DTYPE = {
        'station_id': str, 'num_bikes_available': 'Int16', 'num_ebikes_available': 'Int16',
        'num_bikes_disabled': 'Int16', 'num_docks_available': 'Int16',
        'num_docks_disabled': 'Int16', 'is_installed': 'Int16', 'is_renting': 'Int16',
        'is_returning': 'Int16', 'station_status_last_reported': 'Int64',
        'station_name': str, 'lat': float, 'lon': float, 'region_id': str,
        'capacity': 'Int16', 'has_kiosk': 'boolean',
        'station_information_last_updated': 'Int64', 'missing_station_information': 'boolean',
    }
    def read_csv(path):
        df = pd.read_csv(path, sep=',', na_values='\\N', dtype=DTYPE)
        df['station_status_last_reported'] = pd.to_datetime(
            df['station_status_last_reported'], unit='s', origin='unix', utc=True
        ).dt.tz_convert('US/Eastern')
        return df
    df = pd.concat([read_csv(f) for f in local_files], ignore_index=True)

    # Filter: 2021 only, active stations
    df = df[(df['station_status_last_reported'] >= '2021-01-01') &
            (df['station_status_last_reported'] <  '2022-01-01')]
    df = df[df['missing_station_information'] == False]
    df = df[df['is_installed'] == 1]
    df = df[df['is_renting']   == 1]
    df = df.rename(columns={'station_status_last_reported': 'datetime'})
    df = df.sort_values(['station_id', 'datetime']).reset_index(drop=True)

    # Resample to 30-min intervals
    df = df.set_index('datetime')
    resampled = (
        df.groupby('station_id').resample('30min')
        .agg(num_bikes_available=('num_bikes_available', 'mean'),
             num_docks_available=('num_docks_available', 'mean'),
             capacity=('capacity', 'first'), region_id=('region_id', 'first'),
             lat=('lat', 'first'), lon=('lon', 'first'))
        .reset_index()
    )

    # Drop low-quality stations (>30% missing)
    station_missing = (
        resampled.groupby('station_id')['num_bikes_available']
        .apply(lambda x: x.isnull().mean()).rename('missing_rate')
    )
    good_stations = station_missing[station_missing <= 0.30].index
    df_clean = resampled[resampled['station_id'].isin(good_stations)].copy()

    # Impute: forward fill + backward fill
    df_clean = df_clean.sort_values(['station_id', 'datetime'])
    for col in ['num_bikes_available', 'num_docks_available']:
        df_clean[col] = (df_clean.groupby('station_id')[col]
                         .transform(lambda x: x.ffill().bfill()))

    # Feature engineering
    df_feat = df_clean.copy()
    df_feat['hour']        = df_feat['datetime'].dt.hour
    df_feat['day_of_week'] = df_feat['datetime'].dt.dayofweek
    df_feat['month']       = df_feat['datetime'].dt.month
    df_feat['is_weekend']  = (df_feat['day_of_week'] >= 5).astype(int)
    df_feat['is_peak']     = (((df_feat['hour'] >= 7) & (df_feat['hour'] <= 9)) |
                               ((df_feat['hour'] >= 17) & (df_feat['hour'] <= 19))).astype(int)
    df_feat['hour_sin']    = np.sin(2 * np.pi * df_feat['hour'] / 24)
    df_feat['hour_cos']    = np.cos(2 * np.pi * df_feat['hour'] / 24)
    df_feat['dow_sin']     = np.sin(2 * np.pi * df_feat['day_of_week'] / 7)
    df_feat['dow_cos']     = np.cos(2 * np.pi * df_feat['day_of_week'] / 7)
    df_feat['month_sin']   = np.sin(2 * np.pi * (df_feat['month'] - 1) / 12)
    df_feat['month_cos']   = np.cos(2 * np.pi * (df_feat['month'] - 1) / 12)
    covid_end = pd.Timestamp('2021-07-01', tz='US/Eastern')
    df_feat['is_covid']    = (df_feat['datetime'] < covid_end).astype(int)
    df_feat['availability_ratio'] = (
        df_feat['num_bikes_available'] / df_feat['capacity'].replace(0, np.nan))
    df_feat['bikes_current'] = df_feat['num_bikes_available']
    for lag in [1, 2, 3, 6, 48, 336]:
        df_feat[f'lag_{lag}'] = (df_feat.groupby('station_id')['num_bikes_available']
                                  .shift(lag))
    df_feat['rolling_mean_6'] = (
        df_feat.groupby('station_id')['num_bikes_available']
        .transform(lambda x: x.shift(1).rolling(6, min_periods=3).mean()))
    df_feat['rolling_std_6'] = (
        df_feat.groupby('station_id')['num_bikes_available']
        .transform(lambda x: x.shift(1).rolling(6, min_periods=3).std()))
    df_feat['target'] = df_feat.groupby('station_id')['num_bikes_available'].shift(-1)
    df_feat = df_feat.dropna(
        subset=['target', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_48', 'lag_336'])

    # 80/20 chronological split
    FEATURE_COLS = [
        'bikes_current', 'num_docks_available', 'capacity', 'availability_ratio',
        'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_48', 'lag_336',
        'rolling_mean_6', 'rolling_std_6',
        'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
        'is_covid', 'lat', 'lon',
    ]
    df_feat = df_feat.sort_values('datetime').reset_index(drop=True)
    df_feat[FEATURE_COLS] = df_feat[FEATURE_COLS].fillna(0)
    split_idx = int(len(df_feat) * 0.8)

    X_train = df_feat.loc[:split_idx - 1, FEATURE_COLS].reset_index(drop=True)
    X_test  = df_feat.loc[split_idx:,     FEATURE_COLS].reset_index(drop=True)
    y_train = df_feat.loc[:split_idx - 1, 'target'].reset_index(drop=True)
    y_test  = df_feat.loc[split_idx:,     'target'].reset_index(drop=True)

    feature_names = FEATURE_COLS
    capacity_map  = df_feat.groupby('station_id')['capacity'].first().astype(int).to_dict()
    lat_map       = df_feat.groupby('station_id')['lat'].first().to_dict()
    lon_map       = df_feat.groupby('station_id')['lon'].first().to_dict()

    print(f'[preprocess-data] Done. Train: {X_train.shape} | Test: {X_test.shape}')
    n_stations = len(lat_map)
    sample_ids = list(lat_map.keys())[:5]
    print(f'[preprocess-data] Unique stations: {n_stations} | Sample IDs: {sample_ids}')
    return {
        **state,
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'feature_names': feature_names,
        'capacity_map':  capacity_map,
        'lat_map':       lat_map,
        'lon_map':       lon_map,
    }

In [6]:
# ── 6. train-models node ───────────────────────────────────────────────────
# use_asymmetric_loss=True  -> QuantileRegressor + XGB/LGBM with custom asym loss
#                               alpha swept over state['alpha_grid']
# use_asymmetric_loss=False -> Ridge + XGB/LGBM with default symmetric MSE
#                               single model set, no alpha sweep

from sklearn.linear_model import Ridge, QuantileRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


def train_models(state: PipelineState) -> PipelineState:
    skill_name = 'train-models'
    print(f'\n[train-models] Skill mode: {skill_store.get_mode(skill_name)}')

    use_asym   = state['use_asymmetric_loss']
    alpha_grid = state['alpha_grid']
    X_tr, y_tr = state['X_train'], state['y_train']
    X_te, y_te = state['X_test'],  state['y_test']

    print(f'[train-models] use_asymmetric_loss={use_asym}')
    if use_asym:
        print(f'[train-models] alpha_grid={alpha_grid}')

    def make_asym_obj(a):
        """Asymmetric loss for XGBoost / LightGBM.

        Gradient: a * residual when overestimating, residual otherwise.
        Hessian:  constant h = 1.0 (approximation).
        """
        def obj(y_true, y_pred):
            residual = y_pred - y_true
            grad = np.where(residual > 0, a * residual, residual)
            hess = np.ones_like(y_pred)  # h = 1 constant approximation
            return grad, hess
        return obj

    def fit_xgb(objective, label=''):
        xgb = XGBRegressor(
            n_estimators=500, learning_rate=0.03, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            device='cuda', verbosity=0, objective=objective,
        )
        try:
            xgb.fit(X_tr, y_tr)
        except Exception:
            print(f'[train-models] XGBoost{label}: cuda unavailable, using cpu.')
            xgb.set_params(device='cpu')
            xgb.fit(X_tr, y_tr)
        return xgb

    def fit_lgbm(objective, label=''):
        lgbm = LGBMRegressor(
            n_estimators=500, learning_rate=0.03, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, random_state=42,
            device='gpu', verbosity=-1, objective=objective,
        )
        try:
            lgbm.fit(X_tr, y_tr)
        except Exception:
            print(f'[train-models] LightGBM{label}: gpu unavailable, using cpu.')
            lgbm.set_params(device='cpu')
            lgbm.fit(X_tr, y_tr)
        return lgbm

    trained_models = {}

    def print_row(name, model, alpha_col=''):
        y_pred = model.predict(X_te)
        rmse = np.sqrt(mean_squared_error(y_te, y_pred))
        r2   = r2_score(y_te, y_pred)
        over = (y_pred > y_te.values).mean()
        print(f'{name:<24} {alpha_col:>5} {rmse:>7.3f} {r2:>7.4f} {over:>8.2%}')

    if use_asym:
        # ── Asymmetric mode: QuantileRegressor + custom-loss XGB/LGBM ────
        # QuantileRegressor: q = 1/(1+alpha) gives identical penalty ratio to the custom XGB/LGBM loss (overestimation cost / underest cost = alpha).
        print(f"\n{'Model':<24} {'Alpha':>5} {'RMSE':>7} {'R2':>7} {'Over%':>8}")
        print('-' * 57)
        for alpha in alpha_grid:
            a_tag = f'_a{alpha}'
            q = 1.0 / (1.0 + alpha)

            qr = QuantileRegressor(quantile=q, alpha=0.0, solver='highs')
            MAX_QR_ROWS = 50_000
            if len(X_tr) > MAX_QR_ROWS:
                _rng = np.random.default_rng(42)
                _idx = _rng.choice(len(X_tr), MAX_QR_ROWS, replace=False)
                qr.fit(X_tr.iloc[_idx], y_tr.iloc[_idx])
            else:
                qr.fit(X_tr, y_tr)
            trained_models[f'quantile{a_tag}'] = qr
            print_row('quantile', qr, f'{alpha:.1f}')

            xgb = fit_xgb(make_asym_obj(alpha), label=f' a={alpha}')
            trained_models[f'xgboost{a_tag}'] = xgb
            print_row('xgboost', xgb, f'{alpha:.1f}')

            lgbm = fit_lgbm(make_asym_obj(alpha), label=f' a={alpha}')
            trained_models[f'lightgbm{a_tag}'] = lgbm
            print_row('lightgbm', lgbm, f'{alpha:.1f}')

            print()

        print(f'[train-models] Done. {len(trained_models)} models '
              f'({len(alpha_grid)} alpha values x 3 types)')

    else:
        # ── Symmetric mode: Ridge + default MSE XGB/LGBM ─────────────────
        print(f"\n{'Model':<24} {'RMSE':>7} {'R2':>7} {'Over%':>8}")
        print('-' * 48)

        ridge = Ridge(alpha=1.0)
        ridge.fit(X_tr, y_tr)
        trained_models['ridge'] = ridge
        print_row('ridge', ridge)

        xgb = fit_xgb('reg:squarederror')
        trained_models['xgboost'] = xgb
        print_row('xgboost', xgb)

        lgbm = fit_lgbm('regression')
        trained_models['lightgbm'] = lgbm
        print_row('lightgbm', lgbm)

        print('\n[train-models] Done. 3 models trained (symmetric MSE).')

    return {**state, 'trained_models': trained_models}


In [7]:
# ── 7. evaluate-models node ────────────────────────────────────────
import time
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def evaluate_models(state: PipelineState) -> PipelineState:
    print(f'\n[evaluate-models] Skill mode: {skill_store.get_mode("evaluate-models")}')
    trained_models     = state['trained_models']
    X_test             = state['X_test']
    y_test             = state['y_test']
    # ── Peak-hour mask ─────────────────────────────────────────────────────
    peak_mask = X_test['is_peak'] == 1
    print(f'[evaluate-models] Peak-hour rows: {peak_mask.sum()} / {len(y_test)}')
    evaluation_results = {}
    all_preds          = {}
    model_names        = list(trained_models.keys())
    # ── Compute metrics per model ───────────────────────────────────────────
    for name, model in trained_models.items():
        y_pred     = model.predict(X_test)
        all_preds[name] = y_pred
        # Full test set
        rmse      = np.sqrt(mean_squared_error(y_test, y_pred))
        mae       = mean_absolute_error(y_test, y_pred)
        r2        = r2_score(y_test, y_pred)
        over_rate = (y_pred > y_test.values).mean()
        # Peak-hour subset
        y_pred_pk = y_pred[peak_mask]
        y_test_pk = y_test[peak_mask]
        if len(y_test_pk) > 0:
            peak_rmse     = np.sqrt(mean_squared_error(y_test_pk, y_pred_pk))
            peak_mae      = mean_absolute_error(y_test_pk, y_pred_pk)
            peak_over     = (y_pred_pk > y_test_pk.values).mean()
        else:
            peak_rmse = peak_mae = peak_over = float('nan')
        evaluation_results[name] = {
            'rmse':                  rmse,
            'mae':                   mae,
            'r2':                    r2,
            'overestimation_rate':   over_rate,
            'peak_rmse':             peak_rmse,
            'peak_mae':              peak_mae,
            'peak_overestimation_rate': peak_over,
        }
        print(
            f'[evaluate-models] {name:12s} | '
            f'RMSE={rmse:.3f}  MAE={mae:.3f}  R²={r2:.4f}  '
            f'Over%={over_rate:.2%}  Peak_RMSE={peak_rmse:.3f}'
        )
    # ── Plotting ────────────────────────────────────────────────────────────
    n = len(model_names)
    fig, axes = plt.subplots(2, n, figsize=(6 * n, 10))
    if n == 1:
        axes = axes.reshape(2, 1)
    for idx, name in enumerate(model_names):
        y_pred    = all_preds[name]
        residuals = y_pred - y_test.values
        # Row 0: Predicted vs Actual
        ax = axes[0, idx]
        ax.scatter(y_test, y_pred, alpha=0.2, s=4, color='steelblue')
        lo, hi = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
        ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect fit')
        ax.set_xlabel('Actual bikes available')
        ax.set_ylabel('Predicted bikes available')
        ax.set_title(f'{name}\nPredicted vs Actual')
        ax.legend(fontsize=8)
        # Row 1: Residual plot
        ax = axes[1, idx]
        ax.scatter(y_pred, residuals, alpha=0.2, s=4, color='darkorange')
        ax.axhline(0, color='r', lw=1.5, linestyle='--')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Residual (Pred − Actual)')
        ax.set_title(f'{name}\nResidual Plot')
    plt.suptitle('Model Evaluation — Citi Bike Availability Prediction', fontsize=14, y=1.01)
    plt.tight_layout()
    eval_plot_path = 'eval_plot.png'
    plt.savefig(eval_plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'[evaluate-models] Plot saved → {eval_plot_path}')
    return {
        **state,
        'evaluation_results': evaluation_results,
        'eval_plot_path':     eval_plot_path,
    }

In [8]:
# ── 8. select-model node ───────────────────────────────────────────
import openai

# Explainability ranking (higher = more explainable)
_EXPLAINABILITY = {
    'ridge':         3,   # Linear — fully interpretable
    'random_forest': 2,   # Tree ensemble — feature importance available
    'lightgbm':      1,   # Gradient boost — harder to explain
    'xgboost':       1,   # Gradient boost — harder to explain
}

def _extract_alpha(model_name: str):
    """Extract alpha value from names like 'xgboost_a3.0'; return None if absent."""
    if '_a' not in model_name:
        return None
    try:
        return float(model_name.rsplit('_a', 1)[-1])
    except ValueError:
        return None

def select_model(state: PipelineState) -> PipelineState:
    print(f'\n[select-model] Skill mode: {skill_store.get_mode("select-model")}')
    trained_models     = state['trained_models']
    evaluation_results = state['evaluation_results']
    X_test             = state['X_test']

    # ── Dimension 3: Inference speed (avg over 5 runs, 1000 rows) ──────────
    inference_times = {}
    sample = X_test.iloc[:1000]
    for name, model in trained_models.items():
        t0 = time.time()
        for _ in range(5):
            model.predict(sample)
        inference_times[name] = (time.time() - t0) / 5
        print(f'[select-model] {name} inference (1000 rows, avg 5x): '
              f'{inference_times[name]*1000:.1f} ms')

    # ── New selection logic ─────────────────────────────────────────────────
    # 1) Symmetric baseline: minimum RMSE among alpha=1.0 models
    alpha_1_models = [
        m for m in evaluation_results.keys()
        if _extract_alpha(m) == 1.0
    ]

    if alpha_1_models:
        baseline_rmse = min(evaluation_results[m]['rmse'] for m in alpha_1_models)
        print(f'[select-model] Baseline RMSE (alpha=1.0 best): {baseline_rmse:.4f}')

        # 2) Keep models within +15% RMSE of baseline
        rmse_threshold = baseline_rmse * 1.15
        survivors = [
            m for m in evaluation_results.keys()
            if evaluation_results[m]['rmse'] <= rmse_threshold
        ]
        print(f'[select-model] RMSE threshold (+15%): {rmse_threshold:.4f}')
        print(f'[select-model] Survivors: {survivors}')

        # Fallback rule: if no highest-alpha model survives, fallback to peak RMSE winner
        max_alpha = max(state.get("alpha_grid", [1.0]))
        alpha_max_survivors = [m for m in survivors if _extract_alpha(m) == max_alpha]
        if not alpha_max_survivors:
            print(f'[select-model][WARNING] No alpha={max_alpha} model remains after RMSE filter; fallback to lowest peak RMSE.')
            best_model = min(
                evaluation_results.keys(),
                key=lambda m: evaluation_results[m]['peak_rmse']
            )
            selection_policy = 'fallback_peak_rmse'
        else:
            # 3) Among survivors, choose lowest overestimation rate
            best_model = min(
                survivors,
                key=lambda m: evaluation_results[m]['overestimation_rate']
            )
            selection_policy = 'rmse_filter_then_min_overestimation'
    else:
        # If alpha-tagged models are not present, fallback to peak RMSE
        print('[select-model][WARNING] No alpha=1.0 models found; fallback to lowest peak RMSE.')
        best_model = min(
            evaluation_results.keys(),
            key=lambda m: evaluation_results[m]['peak_rmse']
        )
        selection_policy = 'fallback_peak_rmse_no_alpha1'

    print(f'[select-model] Selection policy: {selection_policy}')
    print(f'[select-model] Selected model → {best_model}')

    # ── Build metrics summary for LLM ───────────────────────────────────────
    lines = []
    for name, m in evaluation_results.items():
        speed_ms = inference_times.get(name, float('nan')) * 1000
        exp      = _EXPLAINABILITY.get(name, 1)
        lines.append(
            f"- {name}: overall RMSE={m['rmse']:.3f}, MAE={m['mae']:.3f}, "
            f"R²={m['r2']:.4f}, peak-hour RMSE={m['peak_rmse']:.3f}, "
            f"overestimation rate={m['overestimation_rate']:.1%}, "
            f"inference speed={speed_ms:.1f} ms/1000 rows, "
            f"explainability={exp}/3"
        )
    metrics_summary = '\n'.join(lines)

    # ── SKILL.md-governed LLM call (tutorial05 pattern) ───────────────────────
    skill_name_sm = 'select-model'
    skill_content = skill_store.get_content(skill_name_sm)
    print(f'[select-model] Skill mode: {skill_store.get_mode(skill_name_sm)}')
    llm_sm = ChatOpenAI(model='gpt-4o-mini', temperature=0.3)
    human_msg = (
        f"Model evaluation results:{metrics_summary}"
        f"Selected model    : {best_model}"
        f"Selection policy  : {selection_policy}"
        f"Write 2-4 sentences in plain business language explaining why "
        f"{best_model} was chosen. Do not use RMSE/MAE/R² abbreviations. "
        f"Translate performance into operational business impact."
    )
    response = llm_sm.invoke([
        SystemMessage(content=skill_content),
        HumanMessage(content=human_msg),
    ])
    selection_reason = response.content.strip()

    print(f'\n[select-model] ─── Final Decision ───────────────────────────')
    print(f'  Selected model : {best_model}')
    print(f'  Reason         : {selection_reason}')
    print(f'──────────────────────────────────────────────────────────────')
    return {
        **state,
        'selected_model':        best_model,
        'selected_model_object': trained_models[best_model],
        'selection_reason':      selection_reason,
    }

In [9]:
# ── 9. Tool Definitions ───────────────────────────────────────────
from langchain_core.tools import tool

@tool
def adj_stations_tool(station_id: str):
    """
    Search for stations within 600m of the target station_id.
    Returns up to 3 nearby stations with distance and capacity info.
    """
    global result
    if result is None:
        return "Station database not yet loaded. Cannot look up neighbours."
    lat_map = result.get("lat_map", {})
    lon_map = result.get("lon_map", {})
    cap_map = result.get("capacity_map", {})
    if station_id not in lat_map:
        sample = ", ".join(list(lat_map.keys())[:5])
        return f"Station {station_id} not found. Sample valid IDs: {sample}"
    tlat = lat_map[station_id]
    tlon = lon_map.get(station_id, 0.0)
    nearby = []
    for sid, lat in lat_map.items():
        if sid == station_id:
            continue
        lon = lon_map.get(sid)
        if lon is None:
            continue
        dlat = (lat - tlat) * 111000
        dlon = (lon - tlon) * 85000  # cos(40.7 deg) approx
        dist = (dlat**2 + dlon**2) ** 0.5
        if dist <= 600:
            nearby.append((sid, round(dist), cap_map.get(sid, 30)))
    if not nearby:
        return f"No stations found within 600m of Station {station_id}."
    nearby.sort(key=lambda x: x[1])
    lines = [f"Station {s}: {d}m away, capacity={c}" for s, d, c in nearby[:3]]
    return "Nearby stations: " + "; ".join(lines)


@tool
def event_calendar_tool(location: str, date: str):
    """
    Check if there are major events (concerts, sports) at a given date and location.
    Input the location (e.g., 'Madison Square Garden', 'Central Park', 'Barclays Center').
    """
    # ==========================================
    # DETERMINISTIC PROCEDURAL GENERATION
    # ==========================================
    # Combine date and location to create a unique, stable random seed.
    # This guarantees that the exact same query will ALWAYS return the exact same event.
    seed_string = f"{date}_{location.lower()}"
    random.seed(seed_string)

    # Define pools of possible events to make it look realistic
    high_risk_events = [
        "NBA Playoffs", "Global Pop Star Concert",
        "New Year's Eve Celebration", "Major Political Rally"
    ]
    medium_risk_events = [
        "Spring Charity Marathon", "International Food Festival",
        "Tech Industry Conference", "Local Parade"
    ]

    # Generate probabilities based on the stable seed
    event_chance = random.random()

    if event_chance < 0.15:
        # 15% chance to have a high-risk event
        event_name = random.choice(high_risk_events)
        status = f"HIGH RISK: {event_name}. Expect massive bike shortage and severe traffic."
    elif event_chance < 0.35:
        # 20% chance to have a medium-risk event
        event_name = random.choice(medium_risk_events)
        status = f"MEDIUM RISK: {event_name}. Partial road closures and moderate demand spikes."
    else:
        # 65% chance for a normal day
        status = "No major events scheduled. Normal traffic expected."

    # Reset the random seed to avoid affecting other parts of your code (like ML models)
    random.seed()

    return f"[Event Database Query] Date: {date} | Location: {location} | Status: {status}"

print("Tools defined successfully.")
tools = [adj_stations_tool, event_calendar_tool]

Tools defined successfully.


In [10]:
# ── 10. run inference and downstream nodes───────────────────────────────────────────
import os
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

def run_inference(state: PipelineState) -> PipelineState:
    """
    Executes demand prediction based on direct latitude and longitude input.
    Instead of station_id, this node performs:
    1. Spatial Search: Finds the closest coordinates in X_test using Euclidean distance.
    2. Temporal Alignment: Filters records matching the input hour, day of week, and month.
    3. Verification: Ensures the matched station exists in the capacity map for business logic.
    """
    print("\n[run-inference] Skill mode: spatial-temporal-inference")

    # Extract core model and data artifacts
    best_model = state.get("selected_model_object")
    X_test = state.get("X_test")
    feature_names = state.get("feature_names", [])

    # Derive spatial inputs from station_id via lat_map / lon_map
    sid_input = state.get("inference_station_id", "")
    lat_map   = state.get("lat_map", {})
    lon_map   = state.get("lon_map", {})
    target_lat = lat_map.get(sid_input)
    target_lon = lon_map.get(sid_input)

    # Extract temporal constraints
    target_hour = state.get("inference_hour")
    target_dow = state.get("inference_dow")
    target_month = state.get("inference_month")

    # Basic safety check
    if best_model is None or X_test is None or X_test.empty or target_lat is None or target_lon is None:
        print("[run-inference] Error: Missing model, data, or coordinates. Using fallback.")
        return {**state, 'prediction': 5.0, 'prediction_confidence': 'Low', 'data_available': False}

    try:
        # =========================================================
        # 1. SPATIAL SEARCH (KNN Logic)
        # Find the row in X_test that is physically closest to the input lat/lon.
        # =========================================================
        print(f"[run-inference] 🔍 Searching for closest station to ({target_lat}, {target_lon})...")

        # Calculate squared Euclidean distance to all rows in X_test
        distances = (X_test['lat'] - target_lat)**2 + (X_test['lon'] - target_lon)**2

        # Filter X_test to only include the specific station found (using a small tolerance)
        closest_row_idx = distances.idxmin()
        closest_lat = X_test.loc[closest_row_idx, 'lat']
        closest_lon = X_test.loc[closest_row_idx, 'lon']

        station_data = X_test[
            (np.isclose(X_test['lat'], closest_lat, atol=1e-5)) &
            (np.isclose(X_test['lon'], closest_lon, atol=1e-5))
        ]

        # =========================================================
        # 2. TEMPORAL MATCHING
        # Within the matched station's data, find rows matching the requested time.
        # =========================================================
        match_query = station_data.copy()

        if target_hour is not None:
            match_query = match_query[match_query['hour'] == target_hour]
        if target_dow is not None:
            match_query = match_query[match_query['day_of_week'] == target_dow]
        if target_month is not None:
            match_query = match_query[match_query['month'] == target_month]

        if not match_query.empty:
            # Take the most recent relevant record
            sample_features = match_query.iloc[[-1]]
            print(f"[run-inference] ✅ Found exact spatio-temporal match.")
        else:
            # Fallback: Best spatial match but temporal mismatch
            sample_features = station_data.iloc[[-1]]
            print(f"[run-inference] ⚠️ Spatial match found, but no record for the specific time. Using latest available for this location.")

        # =========================================================
        # 3. METADATA RECOVERY
        # Link back to a station_id to retrieve capacity_map info
        # =========================================================
        lat_map = state.get("lat_map", {})
        # Inverse lookup to find which station_id this coordinate belongs to
        matched_station_id = "Unknown"
        for sid, lat_val in lat_map.items():
            if np.isclose(lat_val, closest_lat, atol=1e-5):
                matched_station_id = sid
                break

        print(f"[run-inference] Location identified as Station ID: {matched_station_id}")


        # =========================================================
        # 🔎  DATA AUDIT REPORT (VERIFICATION BLOCK)
        # This block explicitly prints the Target vs. Actual data
        # to guarantee the pipeline extracted the correct row.
        # =========================================================
        actual_lat = sample_features['lat'].iloc[0] if 'lat' in sample_features else "N/A"
        actual_lon = sample_features['lon'].iloc[0] if 'lon' in sample_features else "N/A"
        actual_hour = sample_features['hour'].iloc[0] if 'hour' in sample_features else "N/A"
        actual_dow = sample_features['day_of_week'].iloc[0] if 'day_of_week' in sample_features else "N/A"
        actual_cap = sample_features['capacity'].iloc[0] if 'capacity' in sample_features else "N/A"

        match_status = ("Exact spatio-temporal match" if not match_query.empty
                        else "Spatial match only (temporal fallback)")
        print("\n" + "-"*50)
        print("🔎 DATA AUDIT REPORT: Verification of Extracted Row")
        print("-"*50)
        print(f"Status       : {match_status}")
        print(f"Station ID   : {matched_station_id} (Capacity: {actual_cap})")
        print(f"Coordinates  : Expected ({target_lat}, {target_lon}) -> Found ({actual_lat:.4f}, {actual_lon:.4f})")
        print(f"Time (Hour)  : Expected ({target_hour}) -> Found ({actual_hour})")
        print(f"Time (DOW)   : Expected ({target_dow}) -> Found ({actual_dow})")
        print("-"*50 + "\n")

        # =========================================================
        # 4. PREDICTION
        # =========================================================
        # Prepare feature vector (ensuring only trained columns are used)
        X_input = sample_features[feature_names] if feature_names else sample_features

        pred_val = best_model.predict(X_input)
        final_prediction = max(0.0, float(pred_val[0]))

        # Confidence logic based on rolling volatility
        confidence = "Medium"
        confidence_note = "Demand pattern is moderately stable."
        if 'rolling_std_6' in sample_features.columns:
            std = sample_features['rolling_std_6'].iloc[0]
            if std < 2.0:
                confidence = "High"
                confidence_note = "Consistent demand pattern over the past 6 intervals (90 min)."
            elif std > 4.0:
                confidence = "Low"
                confidence_note = "High demand variability in the past 90 min — verify before acting."

        # Extract current inventory from bikes_current feature (lag_0)
        X_input = sample_features[feature_names] if feature_names else sample_features

        pred_val = best_model.predict(X_input)
        final_prediction = max(0.0, float(pred_val[0]))

        # --- NEW: Safe Inventory Extraction Logic ---
        # 1. Retrieve true station capacity for safe fallback calculation
        capacity_map = state.get("capacity_map", {})
        station_capacity = float(capacity_map.get(matched_station_id, 30.0))

        # 2. Set default fallback to 50% capacity (CRITICAL: NEVER use prediction as fallback)
        current_inv = station_capacity * 0.5

        # 3. Extract real data if available in the matched row
        if 'bikes_current' in sample_features.columns:
            current_inv = max(0.0, float(sample_features['bikes_current'].iloc[0]))

        return {
            **state,
            'prediction':            final_prediction,
            'prediction_confidence': confidence,
            'confidence_note':       confidence_note,
            'current_inventory':     current_inv,
            'inference_station_id':  matched_station_id,
        }

    except Exception as e:
        print(f"[run-inference] Execution Error: {e}")
        return {**state, 'prediction': 5.0, 'prediction_confidence': 'Low',
                'confidence_note': 'Fallback value — data unavailable.',
                'current_inventory': 5.0, 'data_available': False}

def dispatch_advisor(state: PipelineState) -> PipelineState:
    """
    Evaluates the station's health and decides the next action using GPT-4-turbo with pre-fetched tool results.
    """
    print("\n[dispatch-advisor] Skill mode: llm-agent")

    station_id       = state.get("inference_station_id", "Unknown")
    prediction       = state.get("prediction", 0.0)
    capacity_map     = state.get("capacity_map", {})
    capacity         = float(capacity_map.get(station_id, 30))
    confidence       = state.get("prediction_confidence", "Medium")
    conf_note        = state.get("confidence_note", "")
    current_inv = state.get("current_inventory")


    # Availability ratio based on actual current inventory (fixes ratio inconsistency)
    avail_ratio = current_inv / capacity if capacity > 0 else 0.5
    avail_ratio = round(min(max(avail_ratio, 0.0), 1.0), 3)
    print(f"[dispatch-advisor] Station {station_id} | "
          f"current={current_inv:.1f} | predicted={prediction:.1f} | "
          f"capacity={capacity:.0f} | avail_ratio={avail_ratio:.2%}")

    # ── SKILL.md-governed LLM call (tutorial05 pattern) ─────────────────────
    skill_name_da = 'dispatch-advisor'
    try:
        skill_content = skill_store.get_content(skill_name_da)
        print(f'[dispatch-advisor] Skill mode: {skill_store.get_mode(skill_name_da)}')
    except Exception as _se:
        print(f'[dispatch-advisor] Skill load failed: {_se} — using inline fallback')
        skill_content = (
            'You are a Citi Bike dispatch advisor. Evaluate station health and '
            'decide on one action: REBALANCE (demand surge), ESCALATE (critically '
            'low stock < 20%), or MONITOR (steady state). '
            'End your reply with exactly one of: REBALANCE, ESCALATE, or MONITOR.'
        )

    # Call tools manually and pass results into HumanMessage
    try:
        adj_result = adj_stations_tool.invoke({"station_id": station_id})
    except Exception as _e:
        adj_result = f"Tool error: {_e}"
    try:
        from datetime import date as _date
        _today = _date.today().strftime("%Y-%m-%d")
        event_result = event_calendar_tool.invoke({"location": "New York", "date": _today})
    except Exception as _e:
        event_result = f"Tool error: {_e}"

    human_msg = (
        f"Station ID      : {station_id}\n"
        f"Current bikes   : {current_inv:.0f} / {capacity:.0f} (availability {avail_ratio:.0%})\n"
        f"ML forecast t+1 : {prediction:.1f} bikes needed in the next 30 minutes\n"
        f"Forecast quality: {confidence} — {conf_note}\n\n"
        f"[Tool Results]\n"
        f"Nearby stations : {adj_result}\n"
        f"Event calendar  : {event_result}"
    )

    try:
        response = llm.invoke([
            SystemMessage(content=skill_content),
            HumanMessage(content=human_msg),
        ])
        raw_text = response.content.strip()

        # ── Try JSON format first (as specified in skill output format) ──
        decision = None
        reasoning = ""
        import json as _json, re as _re
        try:
            # Handle JSON possibly wrapped in markdown code block
            _json_str = _re.sub(r'^```(?:json)?\s*|\s*```$', '', raw_text, flags=_re.DOTALL).strip()
            _parsed = _json.loads(_json_str)
            decision  = str(_parsed.get('dispatch_decision', '')).lower().strip()
            reasoning = str(_parsed.get('dispatch_reasoning', raw_text)).strip()
            if decision not in ('rebalance', 'escalate', 'monitor'):
                decision = None  # invalid value — fall through to keyword search
        except Exception:
            pass  # not JSON — fall through to keyword search

        # ── Fallback: keyword search in free text ──
        if decision is None:
            text_lower = raw_text.lower()
            if 'rebalance' in text_lower:
                decision = 'rebalance'
            elif 'escalate' in text_lower:
                decision = 'escalate'
            else:
                decision = 'monitor'
            # Strip trailing decision keyword (handles punctuation + markdown)
            reasoning = _re.sub(
                r'\s*\*{0,2}(REBALANCE|ESCALATE|MONITOR)\*{0,2}[.!?\s]*$',
                '', raw_text, flags=_re.IGNORECASE
            ).strip() or raw_text.strip()

    except Exception as e:
        print(f'[dispatch-advisor] LLM Error: {e}')
        decision  = 'monitor'
        reasoning = f'System fallback due to LLM error: {e}'

    print(f"[dispatch-advisor] Final Decision: {decision.upper()}")

    return {
        **state,
        'dispatch_decision':  decision,
        'dispatch_reasoning': reasoning,
    }


def dispatch_router(state: PipelineState) -> str:
    """Routes the graph to the corresponding terminal node."""
    return state.get("dispatch_decision", "monitor")


def rebalance_node(state: PipelineState) -> PipelineState:
    """Terminal Node 1: Drafts a work order with full context."""
    print("\n[rebalance-suggest] Generating work order...")
    station_id   = state.get("inference_station_id", "Unknown")
    prediction   = state.get("prediction", 0.0)
    current_inv  = state.get("current_inventory", 0.0)
    capacity     = float(state.get("capacity_map", {}).get(station_id, 30))
    confidence   = state.get("prediction_confidence", "Medium")
    conf_note    = state.get("confidence_note", "")
    reasoning    = state.get("dispatch_reasoning", "")
    model_name   = state.get("selected_model", "ML model")
    shortfall    = max(0.0, prediction - current_inv)

    urgency_map  = {"High": "within 10 minutes", "Medium": "within 20 minutes",
                    "Low": "verify on-site first"}
    urgency      = urgency_map.get(confidence, "within 20 minutes")

    try:
        skill_content_rb = skill_store.get_content('rebalance')
        print(f'[rebalance] Skill mode: {skill_store.get_mode("rebalance")}')
    except Exception as _se:
        print(f'[rebalance] Skill load failed: {_se} — using inline fallback')
        skill_content_rb = (
            'You are a Citi Bike field operations coordinator. '
            'Draft a concise rebalancing work order: specify source station, '
            'number of bikes to transfer, and timing. Be direct and actionable.'
        )
    human_msg_rb = (
        f"Station needing bikes : {station_id}"
        f"Current bikes on dock : {current_inv:.0f} / {capacity:.0f} capacity"
        f"ML forecast (30 min)  : {prediction:.0f} bikes demand"
        f"Estimated shortfall   : {shortfall:.0f} bikes"
        f"Urgency               : {confidence} confidence — act {urgency}"
        f"Advisor context       : {reasoning[:300]}"
    )
    try:
        response = llm.invoke([
            SystemMessage(content=skill_content_rb),
            HumanMessage(content=human_msg_rb),
        ])
        output_text = (
            f"REBALANCING WORK ORDER\n"
            f"Station: {station_id} | Shortfall: {shortfall:.0f} bikes | Urgency: {confidence}\n"
            f"Confidence note: {conf_note}\n\n"
            f"{response.content}\n\n"
            f"Powered by: {model_name}"
        )
    except Exception as e:
        output_text = (
            f"REBALANCING WORK ORDER\n"
            f"Station {station_id} needs ~{shortfall:.0f} bikes transferred urgently ({urgency}).\n"
            f"Confidence: {confidence} — {conf_note}\nPowered by: {model_name}"
        )

    return {**state, 'business_output': output_text}


def escalate_node(state: PipelineState) -> PipelineState:
    """Terminal Node 2: Drafts an escalation alert with full context."""
    print("\n[human-escalation] Generating alert...")
    station_id   = state.get("inference_station_id", "Unknown")
    prediction   = state.get("prediction", 0.0)
    current_inv  = state.get("current_inventory", 0.0)
    capacity     = float(state.get("capacity_map", {}).get(station_id, 30))
    avail_pct    = (current_inv / capacity * 100) if capacity > 0 else 0
    confidence   = state.get("prediction_confidence", "Medium")
    conf_note    = state.get("confidence_note", "")
    reasoning    = state.get("dispatch_reasoning", "")
    model_name   = state.get("selected_model", "ML model")
    shortfall    = max(0.0, prediction - current_inv)

    severity_map = {"High": "CRITICAL", "Medium": "HIGH", "Low": "ELEVATED"}
    severity     = severity_map.get(confidence, "HIGH")

    try:
        skill_content_es = skill_store.get_content('escalate')
        print(f'[escalate] Skill mode: {skill_store.get_mode("escalate")}')
    except Exception as _se:
        print(f'[escalate] Skill load failed: {_se} — using inline fallback')
        skill_content_es = (
            'You are a Citi Bike operations supervisor. '
            'Draft a critical escalation alert: state the severity, '
            'current stock level, expected demand, and required immediate action.'
        )
    human_msg_es = (
        f"Station in crisis   : {station_id}\n"
        f"Current availability: {current_inv:.0f} bikes ({avail_pct:.0f}% of {capacity:.0f} capacity){'  — CRITICALLY LOW' if avail_pct < 20 else ''}\n"
        f"ML forecast (30 min): {prediction:.0f} bikes demand\n"
        f"Shortfall estimate  : {shortfall:.0f} bikes\n"
        f"Severity            : {severity} (forecast confidence: {confidence})"
        f"\nAdvisor context     : {reasoning[:300]}"
    )
    try:
        response = llm.invoke([
            SystemMessage(content=skill_content_es),
            HumanMessage(content=human_msg_es),
        ])
        output_text = (
            f"ESCALATION ALERT — {severity}\n"
            f"Station: {station_id} | Stock: {avail_pct:.0f}% | Shortfall: {shortfall:.0f} bikes\n"
            f"Confidence: {confidence} — {conf_note}\n\n"
            f"{response.content}\n\n"
            f"This alert requires human review. Powered by: {model_name}"
        )
    except Exception as e:
        output_text = (
            f"ESCALATION ALERT — {severity}\n"
            f"Station {station_id}: only {current_inv:.0f}/{capacity:.0f} bikes ({avail_pct:.0f}%). "
            f"Shortfall: {shortfall:.0f} bikes. Immediate supervisor action required.\n"
            f"Confidence: {confidence} — {conf_note}\nPowered by: {model_name}"
        )

    return {**state, 'business_output': output_text}


def monitor_node(state: PipelineState) -> PipelineState:
    """Terminal Node 3: Station health digest with actual data (no LLM)."""
    print("\n[monitor-only] Building status digest...")
    station_id  = state.get("inference_station_id", "Unknown")
    prediction  = state.get("prediction", 0.0)
    current_inv = state.get("current_inventory", 0.0)
    capacity    = float(state.get("capacity_map", {}).get(station_id, 30))
    confidence  = state.get("prediction_confidence", "Medium")
    conf_note   = state.get("confidence_note", "")
    reasoning   = state.get("dispatch_reasoning", "Station operating normally.")
    model_name  = state.get("selected_model", "ML model")
    avail_pct   = (current_inv / capacity * 100) if capacity > 0 else 0
    remaining   = max(0.0, current_inv - prediction)

    next_check_map = {"High": 30, "Medium": 15, "Low": 10}
    next_check  = next_check_map.get(confidence, 15)

    output_text = (
        f"STATION STATUS: NO ACTION REQUIRED\n"
        f"Station: {station_id}\n"
        f"Current availability: {current_inv:.0f} / {capacity:.0f} bikes ({avail_pct:.0f}%)\n"
        f"ML forecast next 30 min: {prediction:.0f} bikes demand\n"
        f"Projected remaining after demand: {remaining:.0f} bikes\n"
        f"Confidence: {confidence} — {conf_note}\n"
        f"Next recommended check: {next_check} minutes\n"
        f"Context: {reasoning[:200]}\n"
        f"Powered by: {model_name}"
    )
    return {**state, 'business_output': output_text}

In [11]:
# ── 11. StateGraph wiring ───────────────────────────────────────────────────
from langgraph.graph import StateGraph, START, END

builder = StateGraph(PipelineState)

builder.add_node('preprocess-data',  preprocess_data)
builder.add_node('train-models',     train_models)
builder.add_node('evaluate-models',  evaluate_models)
builder.add_node('select-model',     select_model)
builder.add_node('run-inference',    run_inference)
builder.add_node('dispatch-advisor', dispatch_advisor)
builder.add_node('rebalance',        rebalance_node)
builder.add_node('escalate',         escalate_node)
builder.add_node('monitor',          monitor_node)

builder.add_edge(START,             'preprocess-data')
builder.add_edge('preprocess-data', 'train-models')
builder.add_edge('train-models',    'evaluate-models')
builder.add_edge('evaluate-models', 'select-model')
builder.add_edge('select-model',    'run-inference')
builder.add_edge('run-inference',   'dispatch-advisor')

builder.add_conditional_edges('dispatch-advisor', dispatch_router)
builder.add_edge('rebalance', END)
builder.add_edge('escalate', END)
builder.add_edge('monitor', END)

# Compile full training + inference pipeline
app = builder.compile()
pipeline = app  # alias used by query_station for lazy training
print("\n End-to-End Pipeline compiled successfully! Nodes:", list(builder.nodes.keys()))

# ── Inference-only subgraph (reuses trained model, skips training stages) ──
inf_builder = StateGraph(PipelineState)
inf_builder.add_node('run-inference',    run_inference)
inf_builder.add_node('dispatch-advisor', dispatch_advisor)
inf_builder.add_node('rebalance',        rebalance_node)
inf_builder.add_node('escalate',         escalate_node)
inf_builder.add_node('monitor',          monitor_node)
inf_builder.add_edge(START,              'run-inference')
inf_builder.add_edge('run-inference',    'dispatch-advisor')
inf_builder.add_conditional_edges('dispatch-advisor', dispatch_router)
inf_builder.add_edge('rebalance', END)
inf_builder.add_edge('escalate',  END)
inf_builder.add_edge('monitor',   END)
inference_app = inf_builder.compile()
print("Inference-only subgraph compiled. Nodes:", list(inf_builder.nodes.keys()))


 End-to-End Pipeline compiled successfully! Nodes: ['preprocess-data', 'train-models', 'evaluate-models', 'select-model', 'run-inference', 'dispatch-advisor', 'rebalance', 'escalate', 'monitor']
Inference-only subgraph compiled. Nodes: ['run-inference', 'dispatch-advisor', 'rebalance', 'escalate', 'monitor']


# Gradio Interface

In [12]:
import gradio as gr
import folium
import json
import math
import random
from datetime import datetime, timedelta


In [13]:
# Training config — pipeline runs lazily on first Gradio query
_TRAIN_INITIAL_STATE: PipelineState = {
    "inference_station_id": "",
    "inference_hour":       0,
    "inference_dow":        0,
    "inference_month":      1,
    "use_asymmetric_loss":  True,
    "alpha_grid":           [1.0, 3.0],
    "X_train": None, "X_test": None, "y_train": None, "y_test": None,
    "feature_names": [], "capacity_map": {}, "lat_map": {}, "lon_map": {},
    "trained_models": {},
    "evaluation_results": {}, "eval_plot_path": "",
    "selected_model": "", "selected_model_object": None, "selection_reason": "",
    "prediction": 0.0, "prediction_confidence": "", "confidence_note": "",
    "current_inventory": 0.0,
    "dispatch_decision": "", "dispatch_reasoning": "",
    "business_output": "",
}

result = None  # populated on first Gradio query


In [14]:
def build_pipeline_html(active: str = None, done: list = None, msg: str = None) -> str:
    """Horizontal pipeline progress strip shown in the Gradio UI."""
    NODES = [
        ("preprocess-data", "Data Prep"),
        ("train-models",    "Training"),
        ("evaluate-models", "Evaluation"),
        ("select-model",    "Model Select"),
        ("run-inference",   "Inference"),
        ("dispatch-advisor","Dispatch"),
        ("terminal",        "Action"),
    ]
    done = done or []
    items = []
    for key, label in NODES:
        if key == active:
            style = ("background:#10b981;color:white;font-weight:700;"
                     "animation:nb-pulse 1.2s ease-in-out infinite;")
            prefix = "⏳ "
        elif key in done:
            style = "background:#d1fae5;color:#065f46;font-weight:600;"
            prefix = "✓ "
        else:
            style = "background:#f3f4f6;color:#9ca3af;"
            prefix = ""
        items.append(f'<span style="padding:4px 10px;border-radius:12px;font-size:12px;{style}">{prefix}{label}</span>')
        if key != "terminal":
            items.append('<span style="color:#d1d5db;font-size:14px;margin:0 2px">›</span>')
    css = "<style>@keyframes nb-pulse{0%{opacity:1}50%{opacity:.45}100%{opacity:1}}</style>"
    inner = "".join(items)
    msg_html = (
        f'<div style="font-size:12px;color:#666;margin-top:6px;padding:0 2px">{msg}</div>'
        if msg else ""
    )
    return (css +
            '<div style="display:flex;align-items:center;flex-wrap:wrap;gap:3px;' +
            'padding:10px 14px;background:#f9fafb;border-radius:8px;' +
            'border:1px solid #e5e7eb;">' + inner + '</div>' + msg_html)

def ratio_to_color(ratio: float) -> str:
    if ratio < 0.20:
        return "#e74c3c"   # red – critical
    elif ratio < 0.35:
        return "#f39c12"   # amber – warning
    else:
        return "#27ae60"   # green – healthy

def ratio_to_label(ratio: float) -> str:
    if ratio < 0.20:
        return "🔴 Critical – very few bikes available"
    elif ratio < 0.35:
        return "🟡 Low – rebalancing may be needed"
    else:
        return "🟢 Healthy – no action required"

def decision_to_label(decision: str) -> str:
    mapping = {
        "rebalance": "Rebalance suggested",
        "escalate":  "Human escalation required",
        "monitor":   "Monitor only – no action needed",
    }
    return mapping.get(decision, decision)

def decision_to_color(decision: str) -> str:
    mapping = {
        "rebalance": "#f39c12",
        "escalate":  "#e74c3c",
        "monitor":   "#27ae60",
    }
    return mapping.get(decision, "#95a5a6")

def confidence_to_color(confidence: str) -> str:
    mapping = {
        "High":   "#27ae60",
        "Medium": "#f39c12",
        "Low":    "#e74c3c",
    }
    return mapping.get(confidence, "#95a5a6")

def confidence_to_bar(confidence: str) -> str:
    levels = {"High": 3, "Medium": 2, "Low": 1}
    filled = levels.get(confidence, 1)
    color  = confidence_to_color(confidence)

    bars = ""
    for i in range(3):
        bg = color if i < filled else "#e0e0e0"
        bars += f'<div style="width:18px; height:10px; background:{bg}; border-radius:2px; display:inline-block; margin-right:3px;"></div>'
    return bars

def build_map(all_stations: list, selected_station_id: str) -> str:
    if not all_stations:
        return "<p style='color:#888; padding:20px;'>No station data available.</p>"

    # Centre the map on the mean lat/lng of all stations
    avg_lat = sum(s["lat"] for s in all_stations) / len(all_stations)
    avg_lng = sum(s["lng"] for s in all_stations) / len(all_stations)

    m = folium.Map(
        location=[40.7484, -73.9967], #fix in the city center of NY
        zoom_start=12,
        tiles="CartoDB positron",  # clean, minimal basemap
    )

    for station in all_stations:
        color = ratio_to_color(station["ratio"])
        is_selected = (station["id"] == selected_station_id)

        # Larger circle for the queried station
        radius = 14 if is_selected else 8
        weight = 3 if is_selected else 1

        _sid = station["id"]
        popup_html = (
            f'<div style="font-family:sans-serif;font-size:13px;min-width:140px;">' +
            f'<b>Station {_sid}</b><br>' +
            f'Availability: {station["ratio"]:.0%}<br>' +
            f'{ratio_to_label(station["ratio"])}<br>' +
            f'<button onclick="window.setStation(\'{_sid}\')" ' +
            f'style="margin-top:6px;padding:3px 10px;background:#10b981;color:white;' +
            f'border:none;border-radius:4px;cursor:pointer;font-size:12px;">' +
            f'Select Station</button></div>'
        )
        folium.CircleMarker(
            location=[station["lat"], station["lng"]],
            radius=radius,
            color="#000000" if is_selected else color,
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            weight=weight,
            tooltip=folium.Tooltip(
                f"<b>Station {station['id']}</b><br>"
                f"Availability: {station['ratio']:.0%}<br>"
                f"{ratio_to_label(station['ratio'])}"
            ),
            popup=folium.Popup(popup_html, max_width=220),
        ).add_to(m)

    # Legend
    legend_html = """
    <div style="position:fixed; bottom:20px; left:20px; z-index:1000;
                background:white; padding:10px 14px; border-radius:8px;
                box-shadow:0 2px 8px rgba(0,0,0,0.2); font-size:13px;">
      <b>Availability</b><br>
      <span style="color:#27ae60;">&#9679;</span> Healthy (&ge;35%)<br>
      <span style="color:#f39c12;">&#9679;</span> Low (20&ndash;35%)<br>
      <span style="color:#e74c3c;">&#9679;</span> Critical (&lt;20%)<br>
      <span style="font-size:11px; color:#666;">Large circle = selected station</span>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # JS: clicking a popup button sets Gradio station input and auto-refreshes
    set_station_js = """
    <script>
    function setStation(sid) {
        try {
            var d = (window.parent !== window) ? window.parent.document : document;
            var c = d.getElementById('station-id-input');
            if (c) {
                var inp = c.querySelector('input, textarea');
                if (inp) {
                    var setter = Object.getOwnPropertyDescriptor(
                        window.HTMLInputElement.prototype, 'value').set;
                    setter.call(inp, sid);
                    inp.dispatchEvent(new Event('input', {bubbles: true}));
                    inp.dispatchEvent(new Event('change', {bubbles: true}));
                }
            }
            setTimeout(function() {
                var ab = d.getElementById('analyze-btn');
                if (ab) { var btn = ab.querySelector('button'); if (btn) btn.click(); }
            }, 250);
        } catch(e) { console.warn('setStation:', e); }
    }
    window.setStation = setStation;
    </script>
    """
    m.get_root().html.add_child(folium.Element(set_station_js))

    return m._repr_html_()

def build_heatmap_html(all_stations: list) -> str:
    """
    Generate a horizontal bar chart for all sites, sorted by availability from lowest to highest. This allows dispatchers to immediately identify the sites that require the most attention.
    """
    if not all_stations:
        return "<p style='color:#888; padding:20px;'>No data available.</p>"

    sorted_stations = sorted(all_stations, key=lambda s: s["ratio"])

    # JS for inline HTML — uses window.parent for iframe compatibility
    click_js = """
    <script>
    function selectHeatmapStation(sid) {
        var d = (window.parent !== window) ? window.parent.document : document;
        var c = d.getElementById('station-id-input');
        if (c) {
            var inp = c.querySelector('input, textarea');
            if (inp) {
                var proto = (inp.tagName === 'TEXTAREA')
                    ? window.HTMLTextAreaElement.prototype
                    : window.HTMLInputElement.prototype;
                var setter = Object.getOwnPropertyDescriptor(proto, 'value').set;
                setter.call(inp, sid);
                inp.dispatchEvent(new Event('input', {bubbles: true}));
                inp.dispatchEvent(new Event('change', {bubbles: true}));
            }
        }
        setTimeout(function() {
            var ab = d.getElementById('analyze-btn');
            if (ab) { var btn = ab.querySelector('button'); if (btn) btn.click(); }
        }, 250);
    }
    </script>
    """
    rows = ""
    for s in sorted_stations:
        color = ratio_to_color(s["ratio"])
        pct = s["ratio"] * 100
        bar_width = max(4, pct)
        sid_safe = s["id"].replace("'", "\'")
        rows += f"""
        <tr onclick="selectHeatmapStation('{sid_safe}')"
            style="cursor:pointer;" title="Click to load dispatch detail">
          <td style="padding:4px 10px 4px 0; font-size:13px; white-space:nowrap;
                     max-width:180px; overflow:hidden; text-overflow:ellipsis;"
              title="Station {s['id']}">Station {s['id']}</td>
          <td style="width:100%;">
            <div style="background:#eee; border-radius:4px; height:18px;">
              <div style="background:{color}; width:{bar_width}%; height:100%;
                          border-radius:4px; min-width:4px;"></div>
            </div>
          </td>
          <td style="padding-left:8px; font-size:13px; font-weight:600; color:{color};">
            {pct:.0f}%
          </td>
        </tr>"""

    return click_js + f"""
    <div style="max-height:420px; overflow-y:auto; font-family:sans-serif;">
      <table style="width:100%; border-collapse:collapse;">
        <thead>
          <tr style="font-size:12px; color:#888; border-bottom:1px solid #ddd;">
            <th style="text-align:left; padding-bottom:6px;">Station</th>
            <th style="text-align:left; padding-bottom:6px;">Bike availability</th>
            <th style="padding-bottom:6px;"></th>
          </tr>
        </thead>
        <tbody>{rows}</tbody>
      </table>
    </div>
    """


def build_dispatch_table_html(all_stations: list, selected_sid: str = None, station_details: dict = None) -> str:
    """
    Scrollable dispatch summary for all monitored stations.
    Uses <details>/<summary> for the LLM toggle — no JavaScript required,
    works reliably inside Gradio gr.HTML() which sandboxes onclick handlers.
    """
    if not all_stations:
        return "<p style='color:#888;padding:20px;'>No dispatch data available.</p>"

    station_details = station_details or {}
    sorted_stations = sorted(all_stations, key=lambda s: s["ratio"])

    rows = ""
    for s in sorted_stations:
        sid   = s["id"]
        ratio = s["ratio"]
        color = ratio_to_color(ratio)
        if ratio < 0.20:
            action, ac = "ESCALATE", "#e74c3c"
        elif ratio < 0.35:
            action, ac = "REBALANCE",   "#f39c12"
        else:
            action, ac = "MONITOR",        "#27ae60"
        is_sel = (sid == selected_sid)
        bg = "background:#f0fdf4;" if is_sel else ""
        fw = "700" if is_sel else "400"
        pct = ratio * 100

        # Detail content shown when <details> is expanded
        if sid in station_details:
            detail_inner = station_details[sid]
            open_attr = " open" if is_sel else ""
        else:
            detail_inner = None
            open_attr = ""

        sid_js = sid.replace("'", "\\'")
        if detail_inner is not None:
            # Show pre-computed analysis via <details>/<summary>
            llm_cell = (
                f'<details{open_attr} style="margin:0;">'
                f'<summary style="display:inline-block;padding:2px 10px;'
                f'background:#10b981;color:white;border-radius:4px;'
                f'cursor:pointer;font-size:11px;white-space:nowrap;'
                f'list-style:none;-webkit-appearance:none;outline:none;">'
                f'LLM &#9654;</summary>'
                f'<div style="margin-top:6px;padding:12px;background:white;'
                f'border:1px solid #e0e0e0;border-radius:8px;'
                f'box-shadow:0 2px 8px rgba(0,0,0,.12);min-width:260px;">'
                + detail_inner +
                f'</div></details>'
            )
        else:
            # data-station: Gradio 5 strips onclick; handled by event delegation in js=
            llm_cell = (
                f'<button data-station="{sid}"'
                f' style="padding:2px 10px;background:#10b981;color:white;'
                f'border:none;border-radius:4px;cursor:pointer;font-size:11px;'
                f'white-space:nowrap;">LLM &#9654;</button>'
            )

        rows += (
            f'<tr style="{bg}">'
            f'<td style="padding:6px 10px 6px 0;font-size:13px;font-weight:{fw};'
            f'white-space:nowrap;">Station {sid}</td>'
            f'<td style="width:36%;padding:5px 4px;">'
            f'<div style="background:#eee;border-radius:4px;height:14px;">'
            f'<div style="background:{color};width:{max(4, pct):.0f}%;height:100%;'
            f'border-radius:4px;min-width:4px;"></div></div></td>'
            f'<td style="padding:5px 4px;font-size:12px;font-weight:600;color:{color};">'
            f'{pct:.0f}%</td>'
            f'<td style="padding:5px 6px;">'
            f'<span style="background:{ac}20;color:{ac};border:1px solid {ac}40;'
            f'border-radius:10px;padding:2px 8px;font-size:11px;font-weight:700;">'
            f'{action}</span></td>'
            f'<td style="padding:5px 8px;">{llm_cell}</td>'
            f'</tr>'
        )

    n_esc = sum(1 for s in all_stations if s["ratio"] < 0.20)
    n_rb  = sum(1 for s in all_stations if 0.20 <= s["ratio"] < 0.35)

    return (
        '<div style="font-family:sans-serif;">'
        '<div style="font-size:12px;color:#888;margin-bottom:8px;">'
        f'{len(all_stations)} stations\u2003'
        f'<span style="color:#e74c3c;font-weight:600;">{n_esc} escalate</span>\u2003'
        f'<span style="color:#f39c12;font-weight:600;">{n_rb} rebalance</span>\u2003'
        'click LLM &#9654; to show analysis</div>'
        '<div style="max-height:420px;overflow-y:auto;">'
        '<table style="width:100%;border-collapse:collapse;">'
        '<thead><tr style="font-size:12px;color:#888;border-bottom:1px solid #ddd;'
        'position:sticky;top:0;background:white;z-index:1;">'
        '<th style="text-align:left;padding-bottom:6px;">Station</th>'
        '<th style="text-align:left;padding-bottom:6px;">Availability</th>'
        '<th></th>'
        '<th style="padding-bottom:6px;">Action</th>'
        '<th style="padding-bottom:6px;"></th></tr></thead>'
        f'<tbody>{rows}</tbody></table></div></div>'
    )

def _make_compact_inline(sid: str, state: dict) -> str:
    """One-line inline summary shown inside the dispatch table expand for selected station."""
    decision   = state.get("dispatch_decision", "monitor")
    prediction = state.get("prediction", 0.0)
    current_inv= state.get("current_inventory", 0.0)
    confidence = state.get("prediction_confidence", "")
    dec_color  = decision_to_color(decision)
    dec_label  = decision_to_label(decision)
    shortfall  = max(0.0, prediction - current_inv)
    return (
        f"<div style='padding:8px 4px;font-size:13px;'>"
        f"<span style='font-weight:700;color:{dec_color};'>{dec_label}</span> &nbsp;|"
        f" Current: {current_inv:.0f} bikes &nbsp;|"
        f" Forecast: {prediction:.0f} bikes &nbsp;|"
        f" Shortfall: {shortfall:.0f} &nbsp;|"
        f" Confidence: {confidence}"
        f"<div style='margin-top:4px;font-size:12px;color:#888;'>"
        f"See full analysis in the panel above.</div></div>"
    )


def format_dispatch_panel(state: dict) -> str:
    station_id      = state.get("inference_station_id", "Unknown")
    prediction      = state.get("prediction", 0.0)
    capacity_map    = state.get("capacity_map", {})
    capacity        = capacity_map.get(station_id, 30)

    ratio = prediction / capacity if capacity > 0 else 0.0
    ratio = min(max(ratio, 0.0), 1.0)

    decision    = state.get("dispatch_decision", "monitor")
    note        = state.get("business_output", "No recommendation available.")
    reasoning   = state.get("dispatch_reasoning", "")
    model_name  = state.get("selected_model", "ML model")
    ts          = state.get("prediction_timestamp", "")
    confidence  = state.get("prediction_confidence", "Medium")   # From Group B
    conf_note   = state.get("confidence_note",
                            "Based on recent station patterns.")  # From Group B

    color       = ratio_to_color(ratio)
    avail_label = ratio_to_label(ratio)
    dec_label   = decision_to_label(decision)
    dec_color   = decision_to_color(decision)
    conf_color  = confidence_to_color(confidence)
    conf_bars   = confidence_to_bar(confidence)

    reasoning_section = ""
    if reasoning:
        reasoning_section = f"""
        <details style="margin-top:12px;">
          <summary style="font-size:12px; color:#888; cursor:pointer;">
            Show agent reasoning (for supervisors)
          </summary>
          <div style="font-size:12px; color:#555; margin-top:8px;
                      padding:10px; background:#f8f9fa; border-radius:6px;
                      line-height:1.5; white-space:pre-wrap;">{reasoning}</div>
        </details>"""

    return f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                max-width:580px;">

      <div style="background:#f8f9fa; border-radius:10px; padding:18px 22px;
                  border-left:5px solid {color}; margin-bottom:16px;">
        <div style="font-size:12px; color:#888; margin-bottom:4px;">{ts}</div>
        <div style="font-size:18px; font-weight:700; margin-bottom:4px;">
          Station {station_id}
        </div>
        <div style="font-size:14px; color:{color}; font-weight:600;">
          {avail_label}
        </div>
        <div style="font-size:28px; font-weight:800; color:{color}; margin-top:6px;">
          {ratio:.0%} available
        </div>
        <div style="font-size:12px; color:#aaa; margin-top:4px;">
          Predicted bikes: {prediction:.1f} / Capacity: {capacity}
        </div>
      </div>

      <!-- Confidence Interval Card -->
      <div style="background:#fff; border:1px solid #e0e0e0; border-radius:10px;
                  padding:14px 22px; margin-bottom:16px;
                  border-left:5px solid {conf_color};">
        <div style="font-size:12px; color:#888; margin-bottom:6px;">
          FORECAST CONFIDENCE
        </div>
        <div style="display:flex; align-items:center; gap:10px; margin-bottom:4px;">
          {conf_bars}
          <span style="font-size:15px; font-weight:700; color:{conf_color};">
            {confidence}
          </span>
        </div>
        <div style="font-size:13px; color:#555; margin-top:4px;">
          {conf_note}
        </div>
      </div>

      <div style="background:#fff; border:1px solid #e0e0e0; border-radius:10px;
                  padding:18px 22px; border-left:5px solid {dec_color};">
        <div style="font-size:12px; color:#888; margin-bottom:6px;">
          RECOMMENDED ACTION
        </div>
        <div style="font-size:17px; font-weight:700; margin-bottom:12px;
                    color:{dec_color};">
          {dec_label}
        </div>
        <div style="font-size:14px; line-height:1.7; color:#333; white-space:pre-wrap;">
          {note}
        </div>
        {reasoning_section}
      </div>

    </div>
    """

In [15]:
def query_station(station_id: str, query_time: str) -> dict:
    """Lazy-init: trains the full pipeline on first call, then reuses the model."""
    global result
    if result is None:
        print("[query_station] First call — running full training pipeline...")
        result = pipeline.invoke(_TRAIN_INITIAL_STATE)
        print(f"[query_station] Training done. Model: {result['selected_model']}")

    dt = datetime.strptime(query_time, "%Y-%m-%d %H:%M")

    initial_state = {
        "inference_station_id": station_id,
        "inference_hour":       dt.hour,
        "inference_dow":        dt.weekday(),
        "inference_month":      dt.month,
        "use_asymmetric_loss":  True,
        "alpha_grid":           [1.0, 3.0],
        "X_train": result["X_train"],
        "X_test":  result["X_test"],
        "y_train": result["y_train"],
        "y_test":  result["y_test"],
        "feature_names":         result["feature_names"],
        "capacity_map":          result["capacity_map"],
        "lat_map":               result["lat_map"],
        "lon_map":               result["lon_map"],
        "trained_models":        result["trained_models"],
        "evaluation_results":    result["evaluation_results"],
        "eval_plot_path":        result["eval_plot_path"],
        "selected_model":        result["selected_model"],
        "selected_model_object": result["selected_model_object"],
        "selection_reason":      result["selection_reason"],
        "prediction": 0.0, "prediction_confidence": "", "confidence_note": "",
        "current_inventory": 0.0,
        "dispatch_decision": "", "dispatch_reasoning": "",
        "business_output": "",
    }

    return inference_app.invoke(initial_state)


def predict_sampled_stations(query_time: str, n: int = 30) -> list:
    """
    Batch ML inference for n stations.

    Samples station IDs directly from lat_map (every entry has a valid coordinate
    by definition).  run_inference handles the spatial search inside X_test, so
    there is no float-precision matching to go wrong here.
    """
    global result
    if result is None:
        print("[predict_sampled_stations] First call — running full training pipeline...")
        result = pipeline.invoke(_TRAIN_INITIAL_STATE)

    dt      = datetime.strptime(query_time, "%Y-%m-%d %H:%M")
    cap_map = result["capacity_map"]
    lat_map = result["lat_map"]
    lon_map = result["lon_map"]

    # Valid stations: must have lat, lon, known capacity, and non-zero coordinates
    valid_sids = [
        sid for sid in lat_map
        if sid in lon_map and sid in cap_map
        and lat_map[sid] and lon_map[sid]  # exclude zero/null coordinates
    ]

    import random as _rnd
    rng = _rnd.Random(42)
    sampled_sids = rng.sample(valid_sids, min(n, len(valid_sids)))

    base_state = {
        **result,
        "inference_hour":        dt.hour,
        "inference_dow":         dt.weekday(),
        "inference_month":       dt.month,
        "prediction":            0.0,
        "prediction_confidence": "",
        "confidence_note":       "",
        "dispatch_decision":     "",
        "dispatch_reasoning":    "",
        "business_output":       "",
    }

    stations = []
    for sid in sampled_sids:
        inf_state = run_inference({**base_state, "inference_station_id": sid})
        if not inf_state.get("data_available", True):  # skip fallback-only results
            continue
        pred  = max(0.0, inf_state.get("prediction", 0.0))
        cap   = cap_map.get(sid, 30)
        ratio = round(min(pred / cap, 1.0), 3) if cap > 0 else 0.0
        stations.append({
            "id":         sid,
            "ratio":      ratio,
            "lat":        lat_map.get(sid, 0),
            "lng":        lon_map.get(sid, 0),
            "prediction": round(pred, 1),
            "capacity":   cap,
        })
    return stations


In [16]:
_DEFAULT_TIME = "2021-10-14 09:00"   # test set start — change below if needed
_last_sampled: list = []             # persists sampled stations between calls


def run_dispatcher_only(station_id: str, time_str: str):
    """Generator: runs LLM dispatcher for one selected station, skips batch inference."""
    global result, _last_sampled

    t = time_str.strip() if time_str and time_str.strip() else _DEFAULT_TIME
    try:
        dt_eff    = datetime.strptime(t, "%Y-%m-%d %H:%M")
        next_time = (dt_eff + timedelta(minutes=30)).strftime("%Y-%m-%d %H:%M")
    except ValueError:
        t         = _DEFAULT_TIME
        dt_eff    = datetime.strptime(_DEFAULT_TIME, "%Y-%m-%d %H:%M")
        next_time = (dt_eff + timedelta(minutes=30)).strftime("%Y-%m-%d %H:%M")

    sid = station_id.strip() if station_id and station_id.strip() else None

    TRAIN_NODES = ["preprocess-data", "train-models", "evaluate-models", "select-model"]
    ALL_NODES   = TRAIN_NODES + ["run-inference", "dispatch-advisor", "terminal"]

    _no_select = (
        "<div style='padding:14px 18px;background:#f0fdf4;border-radius:8px;"
        "border:1px solid #bbf7d0;font-size:14px;color:#166534;margin-bottom:16px;'>"
        "Click any station on the map or heatmap to run LLM dispatch analysis."
        "</div>"
    )

    if not sid or not _last_sampled or result is None:
        yield (
            _no_select + build_dispatch_table_html(_last_sampled, None, {}),
            build_pipeline_html(None, []),
        )
        return

    # Loading indicator
    yield (
        build_dispatch_table_html(_last_sampled, sid, {}),
        build_pipeline_html("dispatch-advisor", TRAIN_NODES, f"LLM analysis: Station {sid}\u2026"),
    )

    sampled        = list(_last_sampled)
    station_details = {}
    selected_panel  = ""

    try:
        _state          = query_station(sid, t)
        selected_panel  = format_dispatch_panel(_state)
        station_details[sid] = _make_compact_inline(sid, _state)
        _pred  = _state.get("prediction", 0.0)
        _cap   = _state.get("capacity_map", {}).get(sid, 30)
        _ratio = round(min(_pred / _cap, 1.0), 3) if _cap > 0 else 0.0
        sampled = [s for s in sampled if s["id"] != sid]
        sampled.append({
            "id":         sid,
            "ratio":      _ratio,
            "lat":        result["lat_map"].get(sid, 0),
            "lng":        result["lon_map"].get(sid, 0),
            "prediction": round(_pred, 1),
            "capacity":   _cap,
        })
    except Exception as exc:
        print(f"[run_dispatcher_only] LLM error for station {sid}: {exc}")

    if selected_panel:
        dispatch_html = (
            selected_panel
            + "<hr style='margin:16px 0;border:none;border-top:1px solid #e5e7eb;'>"
            + "<div style='font-size:12px;color:#888;margin-bottom:8px;'>"
            + "All monitored stations — click LLM &#9654; or map marker to load analysis"
            + "</div>"
            + build_dispatch_table_html(sampled, sid, station_details)
        )
    else:
        dispatch_html = (
            "<div style='padding:14px 18px;background:#fff3cd;border-radius:8px;"
            "border:1px solid #ffc107;font-size:14px;color:#856404;margin-bottom:16px;'>"
            "LLM analysis failed. Please try again.</div>"
            + build_dispatch_table_html(sampled, sid, {})
        )

    n_risk = sum(1 for s in sampled if s["ratio"] < 0.35)
    yield (
        dispatch_html,
        build_pipeline_html(
            "terminal", ALL_NODES[:-1],
            f"LLM done \u00b7 {sid} \u00b7 {len(sampled)} stations \u00b7 {n_risk} at risk",
        ),
    )


def run_interface(station_id: str, time_str: str):
    """Generator: yields 6-tuples (map, heatmap, dispatch, time, pipeline_html, eval_img)."""
    global result, _last_sampled

    # ── Parse time ────────────────────────────────────────────────────────────
    t = time_str.strip() if time_str and time_str.strip() else _DEFAULT_TIME
    try:
        dt_eff    = datetime.strptime(t, "%Y-%m-%d %H:%M")
        next_time = (dt_eff + timedelta(minutes=30)).strftime("%Y-%m-%d %H:%M")
    except ValueError:
        t         = _DEFAULT_TIME
        dt_eff    = datetime.strptime(_DEFAULT_TIME, "%Y-%m-%d %H:%M")
        next_time = (dt_eff + timedelta(minutes=30)).strftime("%Y-%m-%d %H:%M")

    sid = station_id.strip() if station_id and station_id.strip() else None
    _ph = lambda m: f"<p style='color:#888;padding:20px;text-align:center'>{m}</p>"

    TRAIN_NODES = ["preprocess-data", "train-models", "evaluate-models", "select-model"]
    ALL_NODES   = TRAIN_NODES + ["run-inference", "dispatch-advisor", "terminal"]
    is_first    = result is None

    # ── Initial loading indicator ─────────────────────────────────────────────
    yield (
        _ph("Loading map…"),
        _ph("Loading heatmap…"),
        _ph("Loading dispatch…"),
        next_time,
        build_pipeline_html(
            "preprocess-data" if is_first else "run-inference",
            [],
            "Training full pipeline (~2–3 min)…" if is_first else "Running inference…",
        ),
        None,
    )

    # ── Stage 1: stream training nodes (first run only) ───────────────────────
    if is_first:
        done        = []
        accumulated = dict(_TRAIN_INITIAL_STATE)
        try:
            for chunk in pipeline.stream(_TRAIN_INITIAL_STATE, stream_mode="updates"):
                for node_name, node_output in chunk.items():
                    if node_output:
                        accumulated.update(node_output)
                    done.append(node_name)
                    if node_name in TRAIN_NODES:
                        idx = ALL_NODES.index(node_name) if node_name in ALL_NODES else -1
                        nxt = ALL_NODES[idx + 1] if 0 <= idx + 1 < len(ALL_NODES) else None
                        yield (
                            _ph(f"✓ {node_name} done…"),
                            _ph("…"), _ph("…"),
                            next_time,
                            build_pipeline_html(nxt, done[:], f"Completed: {node_name}"),
                            None,
                        )
        except Exception as exc:
            print(f"[run_interface] pipeline.stream error: {exc} — falling back to invoke")
            accumulated = pipeline.invoke(_TRAIN_INITIAL_STATE)
        result = accumulated

    # ── Stage 2: batch ML inference for map / heatmap ─────────────────────────
    yield (
        _ph("Batch inference for stations…"),
        _ph("…"), _ph("…"),
        next_time,
        build_pipeline_html("run-inference", TRAIN_NODES, "Sampling 30 stations…"),
        None,
    )
    sampled = predict_sampled_stations(t, n=30)
    _last_sampled = sampled  # persist for run_dispatcher_only

    if not sampled:
        yield (
            _ph("No station data for this time slot."),
            _ph("No data."), _ph("No data."),
            next_time, build_pipeline_html(None, []), None,
        )
        return

    # ── Stage 3: build dispatch table (ML data only, no LLM) ─────────────────

    # ── Build dispatch tab content ─────────────────────────────────────────────
    dispatch_html = (
        "<div style='padding:14px 18px;background:#f0fdf4;border-radius:8px;"
        "border:1px solid #bbf7d0;font-size:14px;color:#166534;margin-bottom:16px;'>"
        "Click any station on the map or heatmap to run LLM dispatch analysis."
        "</div>"
        + build_dispatch_table_html(sampled, None, {})
    )

    # ── Final yield ───────────────────────────────────────────────────────────
    map_html     = build_map(sampled, sid or "")
    heatmap_html = build_heatmap_html(sampled)
    n_risk    = sum(1 for s in sampled if s["ratio"] < 0.35)
    eval_path = result.get("eval_plot_path") if result else None

    yield (
        map_html, heatmap_html, dispatch_html,
        next_time,
        build_pipeline_html(
            "terminal", ALL_NODES[:-1],
            f"Live · {t} · {len(sampled)} stations · {n_risk} at risk",
        ),
        eval_path,
    )


In [17]:
with gr.Blocks(
    title="Citi Bike Live Monitor",
    theme=gr.themes.Soft(primary_hue="emerald", secondary_hue="slate"),
    css=(
        ".tab-nav button { font-size: 14px; font-weight: 600; } "
        "iframe { border: none; border-radius: 8px; } "
        "tr:hover td { background: #f0fdf4 !important; }"
    ),
    js="""() => {
    function _clickAnalyze(d) {
        var ab = d.getElementById('analyze-btn');
        if (ab) { var btn = ab.querySelector('button'); if (btn) btn.click(); }
    }
    function _setStation(d, sid) {
        var c = d.getElementById('station-id-input');
        if (!c) return;
        var inp = c.querySelector('input, textarea');
        if (!inp) return;

        inp.focus();
        inp.value = sid;
        inp.dispatchEvent(new InputEvent('input', {
            bubbles: true, inputType: 'insertText', data: sid
        }));
        inp.dispatchEvent(new Event('change', {bubbles: true}));
    }
    document.addEventListener("click", function(e) {
        var btn = e.target.closest ? e.target.closest("[data-station]") : null;
        if (!btn) return;
        var sid = btn.getAttribute("data-station");
        if (!sid) return;
        _setStation(document, sid);
        setTimeout(function() { _clickAnalyze(document); }, 350);
    }, true);
    window.runDispatch = function(sid) {
        _setStation(document, sid);
        setTimeout(function() { _clickAnalyze(document); }, 350);
    };
    window.selectStation = window.runDispatch;
}""",
) as demo:

    gr.Markdown("# Citi Bike Live Dispatch Monitor")

    with gr.Row():
        station_input = gr.Textbox(
            label="Station ID",
            placeholder="e.g. 66  — or click LLM ▶ in the Dispatch table",
            value="",
            elem_id="station-id-input",
            scale=3,
        )
        analyze_btn = gr.Button(
            "Show LLM Analysis",
            variant="secondary",
            elem_id="analyze-btn",
            scale=1,
        )

    with gr.Row():
        time_input = gr.Textbox(
            label="Time",
            value=_DEFAULT_TIME,
            scale=3,
        )
        run_btn = gr.Button("Refresh", variant="primary", scale=1)

    pipeline_status = gr.HTML(
        value=build_pipeline_html(None, []),
        label="Pipeline Status",
    )

    with gr.Tabs():
        with gr.Tab("Station map"):
            map_out = gr.HTML(
                value="<p style='color:#aaa; padding:20px;'>Loading…</p>"
            )
        with gr.Tab("Risk heatmap"):
            heatmap_out = gr.HTML(
                value="<p style='color:#aaa; padding:20px;'>Loading…</p>",
            )
        with gr.Tab("Dispatch detail"):
            dispatch_out = gr.HTML(
                value=(
                    "<div style='color:#888;padding:20px;font-size:14px;'>"
                    "Click <b>Refresh</b> to load dispatch suggestions for all stations.</div>"
                )
            )
        with gr.Tab("Model Evaluation"):
            eval_out = gr.Image(
                label="Evaluation Plots (predicted vs actual + residuals)",
                show_label=True,
            )

    gr.Examples(
        examples=[
            ["2021-10-14 08:30"],
            ["2021-10-14 18:00"],
            ["2021-10-14 02:00"],
            ["2021-07-04 12:00"],
        ],
        inputs=[time_input],
        label="⚡ Edge case scenarios (peak AM / peak PM / off-peak / holiday)",
    )

    _outputs = [
        map_out, heatmap_out, dispatch_out,
        time_input, pipeline_status, eval_out,
    ]

    run_btn.click(
        fn=run_interface, inputs=[station_input, time_input],
        outputs=_outputs,
    )
    analyze_btn.click(
        fn=run_dispatcher_only, inputs=[station_input, time_input],
        outputs=[dispatch_out, pipeline_status],
    )
    time_input.submit(
        fn=run_interface, inputs=[station_input, time_input],
        outputs=_outputs,
    )
    demo.load(
        fn=run_interface, inputs=[station_input, time_input],
        outputs=_outputs,
    )

/tmp/ipykernel_5392/65022652.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_5392/65022652.py:1: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_5392/65022652.py:1: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(


In [ ]:
demo.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cddbeb153ff1ab5d84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



[preprocess-data] Skill mode: organisational
[preprocess-data] Sampled: ['citi_bike_data_00040.csv', 'citi_bike_data_00007.csv', 'citi_bike_data_00001.csv', 'citi_bike_data_00047.csv', 'citi_bike_data_00017.csv']
[preprocess-data] Done. Train: (356383, 26) | Test: (89096, 26)
[preprocess-data] Unique stations: 28 | Sample IDs: ['2003', '2012', '223', '229', '247']

[train-models] Skill mode: organisational
[train-models] use_asymmetric_loss=True
[train-models] alpha_grid=[1.0, 3.0]

Model                    Alpha    RMSE      R2    Over%
---------------------------------------------------------
quantile                   1.0   2.197  0.9797   48.61%
xgboost                    1.0   2.030  0.9827   49.79%
lightgbm                   1.0   2.025  0.9827   50.31%

quantile                   3.0   2.409  0.9756   24.12%
xgboost                    3.0   2.253  0.9786   28.64%
lightgbm                   3.0   2.212  0.9794   29.82%

[train-models] Done. 6 models (2 alpha values x 3 types)

[